# Faster R-CNN 베이스라인

**실행 순서:** 0번 환경·Drive 준비 → 1번 라벨 준비 → 2번 스모크 테스트 → 3번 학습과 4번 제출.
0.2에서 패키지가 변경되면 **Colab 서버는 유지하고 커널만 재시작**한 뒤 0번부터 다시 실행한다.

**이번 실험 순서:** 사전학습 없음 seed 0·1·2 학습 → 4번에서 CSV 3개 생성·Kaggle 제출 →
COCO 사전학습 seed 0·1·2 학습 → 4번에서 CSV 3개 생성·Kaggle 제출 → 두 조건의 결과 비교.
총 6회 학습·6회 제출이며, 각 학습 셀은 한 개씩 실행한다. **Run All은 사용하지 않는다.**

환경·패키지 관리는 [README](../../README.md)의 Dependencies,
커밋·push는 [CONTRIBUTING](../../CONTRIBUTING.md)을 따른다.


## 0. Colab 실행환경과 Drive 준비

**목적:** GitHub에서 관리하는 설정으로 학습 패키지 버전을 맞추고, 실제 노트북 커널의 GPU와 Drive 접근을 확인한다.

VSCode의 **기존 Colab 서버 → Python 3 (ipykernel)** 커널에서 실행한다.
전용 `pill-detection-uv` 커널은 이 실행환경에서 연결 오류가 있어 사용하지 않는다.
코드·버전 설정은 GitHub, 데이터와 학습 결과는 개인 Drive에서 관리한다.


### 0.1 GitHub 저장소와 기준 파일 확인

**목적:** 이슈 #23의 작업 브랜치와 `pyproject.toml`·`uv.lock`·`.python-version`을 Colab에서 읽는다.
저장소가 이미 있으면 다시 다운로드하지 않고 브랜치·커밋을 확인한다.


In [ ]:
from pathlib import Path
import hashlib
import importlib
import importlib.metadata as metadata
import os
import shutil
import subprocess
import sys
import tomllib

REPO_URL = "https://github.com/codeit-ai-15th/pill-object-detection.git"
REPO_BRANCH = "exp/23-faster-rcnn-baseline"
REPO_DIR = Path("/content/pill-object-detection")

if not Path("/content").is_dir() or sys.platform != "linux":
    raise RuntimeError("Colab VM이 아닙니다. VSCode에서 Colab의 Python 커널을 선택한 뒤 실행하세요.")
if Path(sys.prefix).resolve() == (REPO_DIR / ".venv").resolve():
    raise RuntimeError("전용 가상환경 커널이 선택됐습니다. 같은 Colab 서버의 기본 Python 3 (ipykernel)로 전환하세요.")

if not REPO_DIR.exists():
    clone_result = subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
        capture_output=True, text=True,
    )
    if clone_result.returncode:
        raise RuntimeError(
            "저장소를 가져오지 못했습니다. GitHub 브랜치·접근 권한을 확인한 뒤 다시 실행하세요.\n"
            + clone_result.stderr.strip()
        )

branch = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True,
).strip()
if branch != REPO_BRANCH:
    raise RuntimeError(f"현재 브랜치는 {branch!r}입니다. Colab 저장소에서 {REPO_BRANCH} 브랜치로 전환한 뒤 실행하세요.")
for name in ["pyproject.toml", "uv.lock", ".python-version"]:
    if not (REPO_DIR / name).is_file():
        raise FileNotFoundError(f"{name}이 없습니다. GitHub 작업 브랜치에 설정 파일을 커밋·push한 뒤 저장소를 갱신하세요.")

repo_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True,
).strip()
print(f"기준 저장소: {branch} | commit {repo_commit}")
print("버전 설정 파일 3개: 확인")


**결과 확인:** 브랜치는 `exp/23-faster-rcnn-baseline`이어야 한다.
GitHub의 설정을 갱신했다면 Colab 저장소에서도 `git pull --ff-only`로 받은 뒤 0.2를 실행한다.


### 0.2 uv.lock 기준 학습 패키지 적용

**목적:** 현재 Colab Python에 팀이 고정한 학습 패키지를 적용한다. `pip install`이나 버전 파일의 수동 수정은 사용하지 않는다.

`--locked`는 잠금 파일 변경을 막는다. Colab 기본 커널을 유지하기 위해 `--no-dev`로 개발용 커널 패키지를 제외하고,
`--inexact`로 프로젝트 밖의 Colab 패키지를 보존한다. 학습 패키지는 다음 셀에서 잠금 버전과 다시 대조한다.


In [ ]:
UV_BIN = shutil.which("uv")
local_uv = Path.home() / ".local/bin/uv"
if UV_BIN is None:
    if not local_uv.is_file():
        installer = subprocess.run(
            ["bash", "-o", "pipefail", "-c", "curl -LsSf https://astral.sh/uv/install.sh | sh"],
            capture_output=True, text=True,
        )
        if installer.returncode:
            raise RuntimeError("uv 설치에 실패했습니다. README의 uv 설치 방법과 네트워크를 확인하세요.\n" + installer.stderr)
    UV_BIN = str(local_uv)

packages_before = {dist.metadata["Name"].lower(): dist.version for dist in metadata.distributions() if dist.metadata["Name"]}
print("uv.lock 기준으로 현재 Colab 환경을 맞추는 중…", flush=True)
sync_result = subprocess.run(
    [UV_BIN, "sync", "--locked", "--no-dev", "--inexact", "--python", sys.executable],
    cwd=REPO_DIR,
    env={**os.environ, "UV_PROJECT_ENVIRONMENT": sys.prefix},
    capture_output=True, text=True,
)
if sync_result.returncode:
    raise RuntimeError(
        "uv sync에 실패했습니다. 아래 오류와 README의 Dependencies를 확인하세요. 잠금 파일을 임의로 바꾸지 마세요.\n"
        + sync_result.stderr.strip()
    )
importlib.invalidate_caches()
packages_after = {dist.metadata["Name"].lower(): dist.version for dist in metadata.distributions() if dist.metadata["Name"]}
ENV_RESTART_REQUIRED = globals().get("ENV_RESTART_REQUIRED", False) or packages_before != packages_after
print("uv 환경 적용: 완료")
print("커널 재시작 필요:", ENV_RESTART_REQUIRED)
if ENV_RESTART_REQUIRED:
    print("상단 Restart로 커널만 재시작한 뒤 0.1부터 다시 실행하세요. Colab 서버를 새로 만들 필요는 없습니다.")


**결과 확인:** `커널 재시작 필요: True`이면 여기서 멈추고 커널을 재시작한다.
같은 서버에서 0번을 다시 실행하면 이미 설치된 패키지를 재사용한다.


### 0.3 실제 학습 환경과 GPU 검증

**목적:** 다른 가상환경의 검사 결과가 아니라, 이 노트북이 실제로 가져오는 패키지 버전과 GPU 연산을 확인한다.
잠금 버전 불일치나 GPU 오류가 있으면 학습을 중단한다.


In [ ]:
from packaging.markers import Marker
from packaging.requirements import Requirement
from packaging.specifiers import SpecifierSet
from packaging.version import Version
import platform

def validate_training_environment(check_gpu_ops=False):
    """노트북의 학습 패키지를 uv.lock과 대조한다.

    Args:
        check_gpu_ops (bool): GPU NMS 연산도 실행할지 여부.

    Returns:
        dict: Python·패키지·CUDA·GPU·잠금 파일의 재현 정보.
    """
    if globals().get("ENV_RESTART_REQUIRED", False):
        raise RuntimeError("패키지가 변경됐습니다. 커널을 재시작하고 0.1부터 다시 실행하세요.")
    project = tomllib.loads((REPO_DIR / "pyproject.toml").read_text())
    lock = tomllib.loads((REPO_DIR / "uv.lock").read_text())
    python_pin = (REPO_DIR / ".python-version").read_text().strip()
    if not platform.python_version().startswith(python_pin + ".") and platform.python_version() != python_pin:
        raise RuntimeError(f"Python 기준은 {python_pin}, 현재는 {platform.python_version()}입니다. 해당 Python의 Colab 런타임을 선택하세요.")
    if Version(platform.python_version()) not in SpecifierSet(project["project"]["requires-python"]):
        raise RuntimeError("Python이 pyproject.toml 조건과 다릅니다. 팀 기준 런타임으로 연결하세요.")

    names = {Requirement(spec).name for spec in project["project"]["dependencies"]} | {"pandas"}
    actual_versions = {}
    mismatches = []
    for name in sorted(names):
        candidates = [
            package for package in lock["package"]
            if package["name"] == name
            and (not package.get("resolution-markers")
                 or any(Marker(marker).evaluate() for marker in package["resolution-markers"]))
        ]
        expected = {package["version"] for package in candidates}
        if len(expected) != 1:
            raise RuntimeError(f"{name}의 잠금 버전을 결정할 수 없습니다. 팀 uv.lock의 플랫폼 조건을 확인하세요.")
        try:
            actual = metadata.version(name)
        except metadata.PackageNotFoundError:
            actual = "미설치"
        actual_versions[name] = actual
        if actual == "미설치" or Version(actual) != Version(next(iter(expected))):
            mismatches.append(f"{name}: 현재 {actual}, 기준 {next(iter(expected))}")
    if mismatches:
        raise RuntimeError("패키지 버전이 다릅니다. 0.2를 실행하고 커널을 재시작하세요.\n" + "\n".join(mismatches))

    for package_name, module_name in [
        ("numpy", "numpy"), ("pandas", "pandas"), ("pillow", "PIL"),
        ("matplotlib", "matplotlib"), ("torchmetrics", "torchmetrics"), ("wandb", "wandb"),
    ]:
        loaded_version = getattr(sys.modules.get(module_name), "__version__", None)
        if loaded_version and Version(loaded_version) != Version(actual_versions[package_name]):
            raise RuntimeError(f"메모리에 이전 {module_name}이 남아 있습니다. 커널을 재시작하고 0번부터 실행하세요.")

    import torch
    import torchvision
    if Version(torch.__version__) != Version(actual_versions["torch"]) or Version(torchvision.__version__) != Version(actual_versions["torchvision"]):
        raise RuntimeError("메모리에 이전 torch가 남아 있습니다. 커널을 재시작하고 0번부터 실행하세요.")
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA를 사용할 수 없습니다. Colab의 GPU 서버에 연결한 뒤 0번을 다시 실행하세요.")
    if check_gpu_ops:
        from torchvision.ops import nms
        boxes = torch.tensor([[0, 0, 10, 10]], dtype=torch.float32, device="cuda")
        scores = torch.tensor([0.9], device="cuda")
        nms(boxes, scores, 0.5)
        torch.cuda.synchronize()
    return {
        "python": platform.python_version(),
        "python_executable": sys.executable,
        "packages": actual_versions,
        "torch_cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "uv_lock_sha256": hashlib.sha256((REPO_DIR / "uv.lock").read_bytes()).hexdigest(),
        "environment_repository_commit": subprocess.check_output(
            ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True,
        ).strip(),
    }

RUNTIME_ENVIRONMENT = validate_training_environment(check_gpu_ops=True)
import torch
import torchvision
print("Python:", RUNTIME_ENVIRONMENT["python"])
print("torch / torchvision:", torch.__version__, "/", torchvision.__version__)
print("CUDA / GPU:", torch.version.cuda, "/", RUNTIME_ENVIRONMENT["gpu"])
print("학습 패키지 잠금 버전 일치 · GPU NMS: 정상")


**결과 확인:** 잠금 버전과 GPU 검사를 통과해야 1번을 진행한다.
개발용 커널·Colab 보조 패키지는 Colab 환경에 남아 있고, 학습에 쓰는 패키지는 위 검사로 고정한다.


### 0.4 Drive 연결과 입출력 경로

**목적:** 원본 데이터와 수정 라벨을 읽을 위치, 학습 결과를 저장할 위치를 지정한다.
`data/raw`는 읽기만 하고, 학습 결과는 Drive의 `runs/faster_rcnn`에 저장한다.


In [ ]:
import sys
from pathlib import Path

# 이미 연결된 Drive는 재인증하지 않는다.
if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/코드잇 ;)/Project1_pill-object-detection"
)
DATA_DIR = DRIVE_PROJECT_DIR / "data"
DRIVE_TRAIN_IMAGE_DIR = DATA_DIR / "raw" / "train_images"
TRAIN_IMAGE_DIR = DRIVE_TRAIN_IMAGE_DIR
ANNOTATION_DIR = DATA_DIR / "raw" / "train_annotations"
SPLIT_PATH = DATA_DIR / "processed" / "split.csv"
FIXES_PATH = DATA_DIR / "processed" / "label_fixes.csv"
RUNS_ROOT = DRIVE_PROJECT_DIR / "runs" / "faster_rcnn"

print("Python:", sys.version.split()[0])
print("현재 작업 폴더:", Path.cwd())
print("Colab 런타임:", Path("/content").exists())
print("Drive 연결:", Path("/content/drive/MyDrive").exists())
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nDrive 프로젝트 경로:", DRIVE_PROJECT_DIR)
for label, target_path in [
    ("train_images", TRAIN_IMAGE_DIR),
    ("train_annotations", ANNOTATION_DIR),
    ("split.csv", SPLIT_PATH),
    ("수정 라벨 CSV", FIXES_PATH),
]:
    print(f"{label}: {'확인' if target_path.exists() else '아직 없음'} | {target_path}")
print("학습 결과 저장 위치:", RUNS_ROOT)


**결과 확인:** train 이미지·JSON·`split.csv`·`label_fixes.csv` 경로가 모두 확인되어야 한다.
원본 데이터는 수정하지 않고 수정 라벨을 메모리에 적용한다.


## 1. 학습 데이터 준비

**목적:** Drive의 원본 JSON에서 클래스와 라벨을 읽고, 팀에서 고정한 split.csv와 수정 라벨을 적용한 뒤 Faster R-CNN 입력 형식으로 변환한다.

원본 JSON·이미지는 변경하지 않는다. 수정 라벨은 메모리에서 반영하며, Drive의 split.csv는 읽기만 한다. 수정 라벨 CSV가 없거나 형식이 맞지 않으면 학습 데이터 준비를 중단한다.


### 1.1 경로와 실행 설정

Drive에 복사한 원본 이미지·JSON, 분할표, 수정 라벨 CSV 경로를 확인한다.


In [ ]:
from pathlib import Path
import ast
import csv
import json
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image
from pill_detection.utils import draw_boxes, xyxy_to_xywh
import matplotlib.pyplot as plt

# 0번에서 연결한 개인 Drive 프로젝트 경로를 사용한다.
DATA_SOURCE = "Google Drive (Colab)"
SPLIT_SEED = 410  # 공유된 분할표의 기준 시드 기록용

required_paths = {
    "train_images": TRAIN_IMAGE_DIR,
    "train_annotations": ANNOTATION_DIR,
    "split.csv": SPLIT_PATH,
    "label_fixes.csv": FIXES_PATH,
}
missing = [f"{name}: {target}" for name, target in required_paths.items() if not target.exists()]
if missing:
    raise FileNotFoundError(
        "학습 입력 파일을 찾지 못했습니다. 아래 자료를 Drive 프로젝트 폴더의 "
        "data/raw 또는 data/processed에 복사한 뒤 다시 실행하세요.\n- "
        + "\n- ".join(missing)
    )

assert TRAIN_IMAGE_DIR.is_dir(), f"학습 이미지 폴더가 아닙니다: {TRAIN_IMAGE_DIR}"
assert ANNOTATION_DIR.is_dir(), f"JSON 폴더가 아닙니다: {ANNOTATION_DIR}"
assert SPLIT_PATH.is_file(), f"분할표 파일이 아닙니다: {SPLIT_PATH}"
assert FIXES_PATH.is_file(), f"수정 라벨 CSV 파일이 아닙니다: {FIXES_PATH}"

print("데이터 출처:", DATA_SOURCE)
print("학습 이미지:", TRAIN_IMAGE_DIR)
print("학습 JSON:", ANNOTATION_DIR)
print("고정 분할표:", SPLIT_PATH)
print("수정 라벨표:", FIXES_PATH)
print("학습 결과 저장 루트:", RUNS_ROOT)


### 1.2 원본 JSON 읽기와 클래스 번호표

이미지 파일명별로 annotation을 모은다. 원본 `category_id`를 오름차순으로 정렬해 클래스 번호를 부여하고, Faster R-CNN에서 사용할 번호(배경 0, 약 클래스 1~56)와 원본 ID 역매핑도 함께 만든다.

In [ ]:
def load_raw_data(annotation_dir):
    """JSON을 읽어 이미지 파일명으로 객체 라벨을 연결한다.

    Args:
        annotation_dir (Path): 원본 JSON 폴더.

    Returns:
        tuple: JSON 경로, 파일명별 이미지 정보, 객체 라벨 목록.
    """
    image_by_name = {}
    annotations = []
    json_paths = sorted(annotation_dir.rglob("*.json"))

    for json_path in json_paths:
        data = json.loads(json_path.read_text(encoding="utf-8"))
        for image_info in data.get("images", []):
            image_by_name[image_info["file_name"]] = image_info
        name_by_image_id = {
            image_info["id"]: image_info["file_name"]
            for image_info in data.get("images", [])
        }
        for ann in data.get("annotations", []):
            annotations.append({
                **ann,
                "file_name": name_by_image_id[ann["image_id"]],
                "category_id": int(ann["category_id"]),
                "bbox": list(map(float, ann["bbox"])),
            })
    return json_paths, image_by_name, annotations

json_paths, image_by_name, raw_annotations = load_raw_data(ANNOTATION_DIR)
raw_annotation_count = len(raw_annotations)
original_category_ids = sorted({ann["category_id"] for ann in raw_annotations})

# EDA/YOLO 번호(0~55)와 Faster R-CNN 번호(1~56)를 함께 관리한다.
class_map = pd.DataFrame({"category_id": original_category_ids})
class_map["class_index_0_based"] = range(len(class_map))
class_map["faster_rcnn_label"] = class_map["class_index_0_based"] + 1
category_to_model_label = dict(zip(class_map["category_id"], class_map["faster_rcnn_label"]))
model_label_to_category = {v: k for k, v in category_to_model_label.items()}

print(f"JSON {len(json_paths)}개 | 이미지 {len(image_by_name)}장 | 원본 bbox {raw_annotation_count}개")
print(f"클래스 수: {len(class_map)}")
display(class_map)


### 1.3 팀에서 고정한 train/val 분할 불러오기

Drive의 split.csv를 그대로 사용한다. 파일명·조합 정보가 원본 이미지 목록과 맞는지, train 185장 / val 47장인지, 같은 약 조합이 두 split에 섞이지 않았는지 검증한다. 이 노트북은 분할표를 새로 생성하거나 덮어쓰지 않는다.


In [ ]:
assert SPLIT_PATH.is_file(), f"Drive에서 split.csv를 찾지 못했습니다: {SPLIT_PATH}"
split_df = pd.read_csv(SPLIT_PATH)

required_split_columns = {"file_name", "combo", "split"}
assert required_split_columns.issubset(split_df.columns), (
    f"split.csv에 필요한 열이 없습니다. 필요: {sorted(required_split_columns)}, "
    f"실제: {list(split_df.columns)}"
)
assert split_df["file_name"].is_unique, "split.csv에 중복 file_name이 있습니다."
assert set(split_df["split"].dropna().unique()) == {"train", "val"}, (
    f"split 값은 train/val이어야 합니다: {split_df['split'].unique()}"
)

image_df = pd.DataFrame({"file_name": sorted(image_by_name)})
image_df["combo"] = image_df["file_name"].str.split("_").str[0]

raw_split_names = set(image_df["file_name"])
provided_split_names = set(split_df["file_name"].astype(str))
assert raw_split_names == provided_split_names, (
    "split.csv와 원본 JSON의 이미지 목록이 다릅니다. "
    f"JSON에만 있음 {len(raw_split_names - provided_split_names)}장, "
    f"split.csv에만 있음 {len(provided_split_names - raw_split_names)}장"
)

split_df["file_name"] = split_df["file_name"].astype(str)
split_df["combo"] = split_df["combo"].astype(str)
split_df["split"] = split_df["split"].astype(str)
expected_combo = image_df.set_index("file_name")["combo"]
provided_combo = split_df.set_index("file_name")["combo"]
assert provided_combo.to_dict() == expected_combo.to_dict(), (
    "split.csv의 combo 값이 이미지 파일명에서 계산한 조합 코드와 다릅니다."
)

image_df = image_df.merge(
    split_df[["file_name", "split"]],
    on="file_name",
    how="left",
    validate="one_to_one",
)
assert image_df["split"].notna().all(), "분할이 배정되지 않은 이미지가 있습니다."
assert image_df["split"].value_counts().to_dict() == {"train": 185, "val": 47}, (
    f"기대 분할은 train 185 / val 47입니다: {image_df['split'].value_counts().to_dict()}"
)
assert image_df.groupby("combo")["split"].nunique().max() == 1, (
    "같은 약 조합이 train과 val 양쪽으로 나뉘었습니다."
)

print("분할표:", SPLIT_PATH)
print(image_df["split"].value_counts().to_string())
print("고유 조합 수:", image_df["combo"].nunique())
print("같은 조합의 train/val 혼합: 없음")


### 1.4 수정 라벨 적용과 분할 연결

Drive의 label_fixes.csv는 file_name, category_id, action, bbox, area 열로 읽는다. add 8건을 추가하고, modify 13건은 파일명과 클래스 ID로 기존 bbox 하나와 연결한다. 이 가운데 bbox가 실제로 달라지는 9건만 bbox를 교체하며, bbox가 같은 4건은 area 단독 수정으로 보고 학습 입력에 반영하지 않는다. 최종 771개 bbox가 되고, 각 bbox는 기존 이미지의 고정 train/val 분할을 따른다.


In [ ]:
def parse_fix_bbox(value, row_number):
    try:
        bbox = [float(v) for v in ast.literal_eval(value)]
    except (ValueError, SyntaxError, TypeError) as exc:
        raise ValueError(f"CSV {row_number}행 bbox를 읽을 수 없습니다: {value!r}") from exc
    if len(bbox) != 4 or not np.isfinite(bbox).all():
        raise ValueError(f"CSV {row_number}행 bbox는 유한한 숫자 4개여야 합니다: {bbox}")
    if bbox[2] <= 0 or bbox[3] <= 0:
        raise ValueError(f"CSV {row_number}행 bbox의 너비/높이는 양수여야 합니다: {bbox}")
    return bbox

with FIXES_PATH.open("r", encoding="utf-8-sig", newline="") as fixes_file:
    fixes_reader = csv.DictReader(fixes_file)
    expected_fix_columns = {"file_name", "category_id", "action", "bbox", "area"}
    actual_fix_columns = set(fixes_reader.fieldnames or [])
    assert expected_fix_columns.issubset(actual_fix_columns), (
        f"label_fixes.csv 필수 열이 없습니다. 필요: {sorted(expected_fix_columns)}, "
        f"실제: {sorted(actual_fix_columns)}"
    )
    fixes = list(fixes_reader)

def apply_fixes(image_by_name, annotations, fixes):
    fixed = [dict(ann) for ann in annotations]
    next_ann_id = max(int(ann.get("id", 0)) for ann in fixed) + 1
    applied = Counter()

    for row_number, row in enumerate(fixes, start=2):
        file_name = row["file_name"].strip()
        category_id = int(row["category_id"])
        action = row["action"].strip().lower()
        bbox_after = parse_fix_bbox(row["bbox"], row_number)
        if file_name not in image_by_name:
            raise KeyError(f"CSV {row_number}행 이미지가 JSON에 없습니다: {file_name}")

        if action == "add":
            image_info = image_by_name[file_name]
            fixed.append({
                "id": next_ann_id,
                "image_id": image_info["id"],
                "file_name": file_name,
                "category_id": category_id,
                "bbox": bbox_after,
                "area": bbox_after[2] * bbox_after[3],
                "iscrowd": 0,
            })
            next_ann_id += 1
            applied["add"] += 1
        elif action == "modify":
            matches = [
                ann for ann in fixed
                if ann["file_name"] == file_name
                and int(ann["category_id"]) == category_id
            ]
            assert len(matches) == 1, (
                f"CSV {row_number}행은 파일명·클래스로 원본 라벨 1개와 "
                f"매칭되어야 합니다. 실제 {len(matches)}개: {file_name}, {category_id}"
            )
            old_bbox = [float(v) for v in matches[0]["bbox"]]
            if np.allclose(old_bbox, bbox_after, rtol=0, atol=1e-6):
                applied["area_only_skipped"] += 1
                continue
            matches[0]["bbox"] = bbox_after
            applied["bbox_modified"] += 1
        else:
            raise ValueError(f"CSV {row_number}행 action은 add/modify여야 합니다: {action!r}")

    return fixed, applied

fixed_annotations, applied_fixes = apply_fixes(image_by_name, raw_annotations, fixes)
fixed_df = pd.DataFrame(fixed_annotations)
fixed_df["split"] = fixed_df["file_name"].map(image_df.set_index("file_name")["split"])

assert sum(row["action"].strip().lower() == "add" for row in fixes) == 8
assert sum(row["action"].strip().lower() == "modify" for row in fixes) == 13
assert applied_fixes["add"] == 8
assert applied_fixes["bbox_modified"] == 9
assert applied_fixes["area_only_skipped"] == 4
assert len(fixed_annotations) == 771
assert fixed_df["split"].isna().sum() == 0
assert fixed_df["category_id"].nunique() == 56
assert set(fixed_df["category_id"]) == set(original_category_ids)

print(f"bbox 추가 {applied_fixes['add']}개 | bbox 수정 {applied_fixes['bbox_modified']}개 | area 단독 수정 건너뜀 {applied_fixes['area_only_skipped']}개")
print("수정 후 bbox:", len(fixed_annotations))
print("분할 미배정 bbox:", int(fixed_df["split"].isna().sum()))


### 1.5 bbox와 클래스 라벨 변환

각 bbox를 `[x, y, width, height]`에서 Faster R-CNN 형식인 `[x_min, y_min, x_max, y_max]`로 바꾼다. 원본 클래스 ID도 미리 만든 번호표를 통해 모델 라벨로 바꾼다.

In [ ]:
from torchvision.ops import box_convert

annotations_by_image = defaultdict(list)
for ann in fixed_annotations:
    info = image_by_name[ann["file_name"]]
    bbox = torch.tensor([ann["bbox"]], dtype=torch.float64)
    x1, y1, x2, y2 = box_convert(bbox, in_fmt="xywh", out_fmt="xyxy")[0].tolist()
    assert x2 > x1 and y2 > y1, f"너비/높이가 0 이하입니다. 수정 라벨표를 확인하세요: {ann}"
    assert x1 >= 0 and y1 >= 0 and x2 <= info["width"] and y2 <= info["height"], (
        f"이미지 경계를 벗어났습니다. 수정 라벨표를 확인하세요: {ann}"
    )
    annotations_by_image[ann["file_name"]].append({
        "bbox_xyxy": [x1, y1, x2, y2],
        "category_id": ann["category_id"],
        "model_label": category_to_model_label[ann["category_id"]],
    })

print("이미지별 라벨 연결 완료:", len(annotations_by_image), "장")
print("변환 bbox 수:", sum(len(v) for v in annotations_by_image.values()))
print("Faster R-CNN 클래스 라벨 범위: 1~", len(class_map), "(0은 배경)")


### 1.6 이미지와 bbox 시각화

train에서 객체 3개 또는 4개가 있는 이미지를 고정 시드로 골라 확인한다. 초록색 bbox는 수정 라벨을 반영한 뒤 모델에 전달할 박스다.

In [ ]:
def show_examples(image_df, annotations_by_image, object_counts=(3, 4)):
    """객체 3개·4개인 이미지의 수정된 정답 박스를 확인한다.

    Args:
        image_df (pd.DataFrame): 이미지와 분할 정보.
        annotations_by_image (dict): 파일명별 모델 입력 라벨.
        object_counts (tuple): 확인할 이미지당 객체 수.
    """
    rng = np.random.default_rng(SPLIT_SEED)
    train_names = image_df.loc[image_df["split"] == "train", "file_name"].tolist()
    selected = []
    for count in object_counts:
        candidates = [name for name in train_names
                      if len(annotations_by_image[name]) == count]
        if candidates:
            selected.append((count, rng.choice(candidates)))

    if not selected:
        print("조건에 맞는 이미지가 없습니다.")
        return

    fig, axes = plt.subplots(1, len(selected), figsize=(7 * len(selected), 8))
    if len(selected) == 1:
        axes = [axes]
    for ax, (count, file_name) in zip(axes, selected):
        objects = annotations_by_image[file_name]
        image = draw_boxes(
            TRAIN_IMAGE_DIR / file_name,
            [xyxy_to_xywh(ann["bbox_xyxy"]) for ann in objects],
            [str(ann["category_id"]) for ann in objects],
            color="lime", width=4,
        )
        ax.imshow(image)
        ax.set_title(f"objects={count} | {file_name}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_examples(image_df, annotations_by_image)


### 1.7 Colab 로컬 디스크에 학습·검증 이미지 준비

Drive의 원본 이미지 232장을 `/content/pill_image_cache`로 한 번 복사해 매 epoch의 Drive 읽기를 줄인다.
픽셀·해상도·파일 내용은 바꾸지 않으며, 복사한 파일의 크기와 SHA-256을 검증한다.
같은 VM에서는 검증된 복사본을 재사용하고, 복사 중 중단되면 다음 실행에서 남은 파일만 복사한다.
VM이 새로 만들어져 캐시가 없어지면 이 셀을 다시 실행한다. 데이터·가중치·결과 원본은 계속 Drive에 보관한다.

첫 복사는 시간이 걸릴 수 있다. `TRAIN_IMAGE_DIR`을 로컬 경로로 바꾼 뒤 2.1에서 Dataset을 새로 만든다.
이 셀은 train과 val을 포함한 학습용 이미지 232장만 준비한다. Test 캐시는 4.2에서 같은 함수로 별도 준비한다.


In [ ]:
import shutil


def file_sha256(path):
    """파일 내용의 SHA-256을 메모리에 전부 올리지 않고 계산한다.

    Args:
        path (Path): 검사할 파일 경로.

    Returns:
        str: 파일 내용의 SHA-256.
    """
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def cache_training_images(source_dir, file_names, cache_root):
    """원본을 변경하지 않고 train/val 이미지를 Colab 디스크에 검증·복사한다.

    Args:
        source_dir (Path): Drive 원본 이미지 폴더.
        file_names (list[str]): JSON과 분할표로 확인한 고유 이미지 파일명.
        cache_root (Path): Colab VM 내부의 캐시 루트.

    Returns:
        tuple: 캐시 이미지 폴더와 이미지 내용 목록의 SHA-256.
    """
    source_dir, cache_root = Path(source_dir), Path(cache_root)
    names = sorted(set(map(str, file_names)))
    if not names:
        raise ValueError("캐시할 이미지가 없습니다. 1.2~1.4의 라벨·분할 연결부터 확인하세요.")
    for name in names:
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"잘못된 상대 이미지 경로입니다. 원본 JSON의 file_name을 확인하세요: {name}")
    source_id = hashlib.sha256(str(source_dir.resolve()).encode()).hexdigest()[:12]
    local_dir = cache_root / source_id
    local_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = local_dir / "cache_manifest.json"
    try:
        previous = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
    except (ValueError, OSError):
        previous = {}

    source_stats = {}
    for name in names:
        source_path = source_dir / name
        if not source_path.is_file():
            raise FileNotFoundError(f"원본 이미지가 없습니다. Drive 업로드를 확인하세요: {source_path}")
        info = source_path.stat()
        source_stats[name] = {"size": info.st_size, "mtime_ns": info.st_mtime_ns}

    entries, pending = {}, []
    previous_files = previous.get("files", {})
    for name in names:
        entry = previous_files.get(name, {})
        local_path = local_dir / name
        if (entry.get("source") == source_stats[name] and local_path.is_file()
                and local_path.stat().st_size == source_stats[name]["size"]
                and file_sha256(local_path) == entry.get("sha256")):
            entries[name] = entry
        else:
            pending.append(name)
    required_bytes = sum(source_stats[name]["size"] for name in pending)
    if pending and shutil.disk_usage(local_dir).free < required_bytes + 512 * 1024 * 1024:
        raise RuntimeError("Colab 디스크 여유 공간이 부족합니다. 불필요한 VM 캐시를 정리한 뒤 1.7을 다시 실행하세요.")
    print(f"이미지 캐시: {len(names)}장 | 재사용 {len(entries)}장 | 복사 {len(pending)}장", flush=True)
    for index, name in enumerate(pending, 1):
        source_path, local_path = source_dir / name, local_dir / name
        local_path.parent.mkdir(parents=True, exist_ok=True)
        temporary_path = local_path.with_name(local_path.name + ".copying")
        digest = hashlib.sha256()
        try:
            # 읽는 동안 해시도 계산해 Drive 원본을 두 번 읽지 않는다.
            with source_path.open("rb") as source, temporary_path.open("wb") as output:
                for block in iter(lambda: source.read(1024 * 1024), b""):
                    output.write(block)
                    digest.update(block)
            after = source_path.stat()
            if {"size": after.st_size, "mtime_ns": after.st_mtime_ns} != source_stats[name]:
                raise RuntimeError(f"복사 중 원본이 변경됐습니다. 업로드가 끝난 뒤 1.7을 다시 실행하세요: {name}")
            if temporary_path.stat().st_size != source_stats[name]["size"] or file_sha256(temporary_path) != digest.hexdigest():
                raise RuntimeError(f"이미지 복사 검증에 실패했습니다. 1.7을 다시 실행하세요: {name}")
            os.replace(temporary_path, local_path)
            entries[name] = {"source": source_stats[name], "sha256": digest.hexdigest()}
        finally:
            temporary_path.unlink(missing_ok=True)
        # 세션이 중단돼도 앞서 검증한 복사본은 다음 실행에서 재사용한다.
        temporary_manifest = manifest_path.with_suffix(".json.tmp")
        temporary_manifest.write_text(json.dumps({"files": entries}, ensure_ascii=False, indent=2))
        os.replace(temporary_manifest, manifest_path)
        if index % 25 == 0 or index == len(pending):
            print(f"복사·검증: {index}/{len(pending)}장", flush=True)
    content_id = hashlib.sha256(json.dumps(
        {name: entries[name]["sha256"] for name in names}, sort_keys=True
    ).encode()).hexdigest()
    return local_dir, content_id


LOCAL_IMAGE_CACHE_ROOT = Path("/content/pill_image_cache")
TRAIN_IMAGE_DIR, IMAGE_CONTENT_SHA256 = cache_training_images(
    DRIVE_TRAIN_IMAGE_DIR, image_df["file_name"].tolist(), LOCAL_IMAGE_CACHE_ROOT,
)
print("학습·검증 이미지 읽기:", TRAIN_IMAGE_DIR)
print("원본 보관:", DRIVE_TRAIN_IMAGE_DIR)
print("결과·체크포인트 저장:", RUNS_ROOT)


### 1단계 확인

- 원본 bbox에 수정표를 반영해 총 771개가 되었는가?
- Drive의 고정 분할표와 원본 이미지가 모두 연결되었는가?
- train 185장 / val 47장이고, 미배정 bbox가 0개인가?
- 시각화에서 수정 bbox가 실제 알약을 감싸는가?
- 1.7에서 고유 이미지 232장의 로컬 복사·검증이 완료됐는가?

이 확인을 통과하면 Dataset/DataLoader와 짧은 Faster R-CNN 스모크 학습을 진행한다.


## 2. 파이프라인 작동 확인

**목적:** 준비한 이미지와 라벨이 Dataset/DataLoader를 거쳐 Faster R-CNN 학습과 검증 추론까지 오류 없이 연결되는지 짧게 확인한다.

이 단계는 점수 비교가 아니라 입력 형식, 손실 계산, 가중치 업데이트, 예측 결과 출력 흐름을 점검한다. 학습은 train 데이터 일부 배치만 사용한다.


### 2.1 공통 설정·Dataset과 DataLoader 구성

**목적:** 팀 공통 조건을 정의하고 실제 16장 배치를 구성한다. 이 셀의 설정을 스모크와 본 학습이 함께 사용한다.
최대 100 epoch, patience 30, batch 16, 이미지 긴 변 제한 640, AdamW, workers 0, AMP 미사용이다.
학습률은 현재 팀 잠금 버전의 YOLO auto AdamW 계산식으로 맞춰 약 56종에서 0.000167을 사용한다.
1.7의 로컬 이미지를 읽으며, 클래스 번호·bbox는 기존과 같다. validation batch는 1이다.
torchvision 객체 검출 모델에는 YOLO식 자동 증강 파이프라인이 없으므로 추가 증강은 넣지 않는다.


In [ ]:
import time
import random

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import to_tensor

# 팀 공통 조건과 Faster R-CNN의 적용값. 수정 시 2.1~2.4부터 다시 확인한다.
TRAIN_SEEDS = [0, 1, 2]
MAX_EPOCHS = 100
EARLY_STOP_MIN_EPOCHS = 20
EARLY_STOP_PATIENCE = 30
EARLY_STOP_MIN_DELTA = 0.001
BASELINE_BATCH_SIZE = 16
BASELINE_IMAGE_SIZE = 640
# 종횡비를 유지하면서 긴 변이 640을 넘지 않게 한다. YOLO의 정사각형 letterbox와는 다르다.
BASELINE_MIN_SIZE = BASELINE_IMAGE_SIZE
BASELINE_MAX_SIZE = BASELINE_IMAGE_SIZE
BASELINE_OPTIMIZER = "AdamW"
# 현재 잠금 버전 Ultralytics auto의 AdamW 초기 lr 식. 약 56종이면 0.000167이다.
BASELINE_LR = round(0.002 * 5 / (4 + len(class_map)), 6)
BASELINE_BETAS = (0.9, 0.999)
BASELINE_WEIGHT_DECAY = 0.0005
BASELINE_SCORE_THRESHOLD = 0.001
CHECKPOINT_EVERY_EPOCHS = 5
DATA_LOADER_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()
USE_AMP = False
MAP_IOU_THRESHOLDS = [0.75, 0.80, 0.85, 0.90, 0.95]
WANDB_PREVIEW_COUNT = 3
WANDB_PREVIEW_SCORE_THRESHOLD = 0.30
WANDB_PREVIEW_MAX_SIDE = 1280
EXPERIMENT_CONDITIONS = {
    "scratch": {"label": "사전학습 없음", "weights": None},
    "coco": {"label": "COCO 사전학습", "weights": "COCO_V1"},
}

# 성능 평가가 아니라 실제 16장 한 배치의 forward/backward/AdamW step을 점검한다.
SMOKE_SEED = 0
SMOKE_BATCH_SIZE = BASELINE_BATCH_SIZE
VAL_PREVIEW_COUNT = 8
SMOKE_BATCH16_PASSED = False
SMOKE_VALIDATED_CONFIGURATION = None

random.seed(SMOKE_SEED)
np.random.seed(SMOKE_SEED)
torch.manual_seed(SMOKE_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SMOKE_SEED)

class PillDetectionDataset(Dataset):
    def __init__(self, image_frame, image_dir, labels_by_image):
        self.frame = image_frame.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.labels_by_image = labels_by_image

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        file_name = self.frame.loc[index, "file_name"]
        with Image.open(self.image_dir / file_name) as image:
            image_tensor = to_tensor(image.convert("RGB"))

        objects = self.labels_by_image[file_name]
        boxes = torch.tensor(
            [obj["bbox_xyxy"] for obj in objects], dtype=torch.float32
        ).reshape(-1, 4)
        labels = torch.tensor(
            [obj["model_label"] for obj in objects], dtype=torch.int64
        )
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor(index, dtype=torch.int64),
            "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
        }
        return image_tensor, target

def detection_collate_fn(batch):
    # 이미지마다 객체 수가 달라 기본 collate 대신 목록으로 묶는다.
    return tuple(zip(*batch))

train_frame = image_df.loc[image_df["split"] == "train", ["file_name"]].copy()
val_frame = image_df.loc[image_df["split"] == "val", ["file_name"]].copy()
train_dataset = PillDetectionDataset(train_frame, TRAIN_IMAGE_DIR, annotations_by_image)
val_dataset = PillDetectionDataset(val_frame, TRAIN_IMAGE_DIR, annotations_by_image)
train_loader = DataLoader(
    train_dataset, batch_size=SMOKE_BATCH_SIZE, shuffle=True,
    num_workers=DATA_LOADER_WORKERS, pin_memory=PIN_MEMORY, collate_fn=detection_collate_fn,
)
val_loader = DataLoader(
    val_dataset, batch_size=1, shuffle=False,
    num_workers=DATA_LOADER_WORKERS, pin_memory=PIN_MEMORY, collate_fn=detection_collate_fn,
)

sample_images, sample_targets = next(iter(train_loader))
assert len(sample_images) == len(sample_targets) == SMOKE_BATCH_SIZE
assert sample_images[0].ndim == 3 and sample_images[0].shape[0] == 3
assert sample_targets[0]["boxes"].shape[1] == 4
assert sample_targets[0]["labels"].dtype == torch.int64
assert (sample_targets[0]["labels"] > 0).all(), "0은 배경 클래스이므로 실제 약 라벨에 쓸 수 없습니다."

print(f"train Dataset/DataLoader: {len(train_dataset)}장 / {len(train_loader)} batch")
print(f"val Dataset/DataLoader: {len(val_dataset)}장 / {len(val_loader)} batch")
print("샘플 이미지 텐서:", tuple(sample_images[0].shape), sample_images[0].dtype)
print("샘플 target:", {key: tuple(value.shape) for key, value in sample_targets[0].items()})

# 데이터에 실제 등장하는 클래스 라벨 수를 split별로 집계한다.
split_model_labels = {}
split_box_counts = {}
for split_name in ("train", "val"):
    split_names = image_df.loc[image_df["split"] == split_name, "file_name"]
    split_objects = [obj for name in split_names for obj in annotations_by_image[name]]
    split_model_labels[split_name] = sorted({obj["model_label"] for obj in split_objects})
    split_box_counts[split_name] = len(split_objects)

print(f"수정 후 전체 객체 라벨: {len(fixed_annotations)}개")
print(f"train 객체/등장 클래스: {split_box_counts['train']}개 / {len(split_model_labels['train'])}종")
print(f"val 객체/등장 클래스: {split_box_counts['val']}개 / {len(split_model_labels['val'])}종")
print(f"모델에 전달하는 실제 클래스 라벨 범위: 1~{len(class_map)} (0은 배경)")
print("첫 샘플의 model_label:", sample_targets[0]["labels"].tolist())
print(f"공통 설정: epochs={MAX_EPOCHS}, patience={EARLY_STOP_PATIENCE}, batch={BASELINE_BATCH_SIZE}, image cap={BASELINE_IMAGE_SIZE}")
print(f"optimizer={BASELINE_OPTIMIZER}, lr={BASELINE_LR}, betas={BASELINE_BETAS}, weight_decay={BASELINE_WEIGHT_DECAY}")
print("증강: torchvision 모델 기본값(추가 랜덤 증강 없음) | workers=0 | AMP=False")
# 샘플 확인용 원본 크기 텐서 16장을 계속 들고 있지 않는다.
del sample_images, sample_targets


**결과 확인:** train/val 이미지 수와 bbox·등장 클래스 수를 확인한다. Faster R-CNN 라벨은 1~56이며 0은 배경이다. 이미지 텐서는 `[채널, 높이, 너비]`, bbox는 `[객체 수, 4]` 형태여야 한다.


### 2.2 본 학습과 같은 모델·optimizer 함수 준비

스모크와 본 학습이 같은 모델 생성 함수를 사용한다. 사전학습 없이 만들지만 COCO 조건과 구조·정규화는 같다.
입력은 종횡비를 유지하고 긴 변을 최대 640으로 제한한다. 모델 내부에서 리사이즈하므로 원본 bbox 좌표를 미리 바꾸지 않는다.
AdamW 함수도 여기서 정의해 스모크와 본 학습에서 공유한다.


In [ ]:
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import FrozenBatchNorm2d

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = len(class_map) + 1


def use_common_normalization(module):
    """두 조건의 BatchNorm 구조와 동작을 COCO 기준으로 통일한다.

    Args:
        module (torch.nn.Module): 정규화 층을 바꿀 모델 또는 하위 모듈.
    """
    for name, child in list(module.named_children()):
        if isinstance(child, torch.nn.BatchNorm2d):
            frozen = FrozenBatchNorm2d(child.num_features, eps=0.0)
            with torch.no_grad():
                frozen.weight.copy_(child.weight)
                frozen.bias.copy_(child.bias)
                frozen.running_mean.copy_(child.running_mean)
                frozen.running_var.copy_(child.running_var)
            setattr(module, name, frozen)
        else:
            use_common_normalization(child)


def build_experiment_model(condition, train_seed):
    """동일한 구조를 만들고 초기 가중치만 조건에 따라 적용한다.

    Args:
        condition (str): scratch 또는 coco.
        train_seed (int): 새 약 분류 헤드의 초기화 seed.

    Returns:
        FasterRCNN: 약 56종과 배경을 출력하는 학습용 모델.
    """
    if condition not in EXPERIMENT_CONDITIONS:
        raise ValueError("초기화 조건은 scratch 또는 coco여야 합니다. 3.3 설정을 확인하세요.")
    # weights만 바꾸면 torchvision이 정규화·백본 고정 범위도 바꾸므로 구조를 먼저 통일한다.
    model = fasterrcnn_resnet50_fpn(
        weights=None, weights_backbone=None, num_classes=91,
        min_size=BASELINE_MIN_SIZE, max_size=BASELINE_MAX_SIZE,
    )
    use_common_normalization(model)
    if condition == "coco":
        print("COCO_V1 가중치 확인·다운로드 중…", flush=True)
        weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1
        try:
            state = weights.get_state_dict(progress=True, check_hash=True)
        except Exception as error:
            raise RuntimeError(
                "COCO 가중치를 받지 못했습니다. Colab 인터넷 연결을 확인한 뒤 이 학습 셀을 다시 실행하세요."
            ) from error
        model.load_state_dict(state, strict=True)
        del state
    # COCO의 91개 출력 헤드는 버리고, 두 조건에 동일한 seed로 새 약 분류·회귀 헤드를 만든다.
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(train_seed)
        model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    for parameter in model.parameters():
        parameter.requires_grad_(True)
    model.roi_heads.score_thresh = BASELINE_SCORE_THRESHOLD
    return model.to(DEVICE)



def create_training_optimizer(model):
    """스모크와 본 학습에 같은 AdamW와 decay 그룹을 적용한다.

    Args:
        model (torch.nn.Module): 학습할 모델.

    Returns:
        torch.optim.AdamW: bias·정규화 파라미터에는 decay를 적용하지 않는 optimizer.
    """
    if BASELINE_OPTIMIZER != "AdamW":
        raise ValueError("현재 optimizer 함수는 AdamW용입니다. 2.1 설정과 함수를 함께 확인하세요.")
    decay, no_decay = [], []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.endswith("bias") or parameter.ndim <= 1:
            no_decay.append(parameter)
        else:
            decay.append(parameter)
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": BASELINE_WEIGHT_DECAY},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=BASELINE_LR, betas=BASELINE_BETAS,
    )


def smoke_configuration():
    """배치 16 확인 시와 본 학습의 설정을 대조할 정보를 반환한다.

    Returns:
        dict: batch·입력·optimizer·정밀도·클래스 수 설정.
    """
    return {
        "batch_size": BASELINE_BATCH_SIZE,
        "min_size": BASELINE_MIN_SIZE, "max_size": BASELINE_MAX_SIZE,
        "optimizer": BASELINE_OPTIMIZER, "lr": BASELINE_LR,
        "betas": list(BASELINE_BETAS), "weight_decay": BASELINE_WEIGHT_DECAY,
        "num_workers": DATA_LOADER_WORKERS, "amp": USE_AMP,
        "num_classes": NUM_CLASSES,
        "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
    }


model = build_experiment_model("scratch", SMOKE_SEED)
print("사용 장치:", DEVICE)
print("모델 출력 클래스 수:", NUM_CLASSES, "(약 56종 + 배경)")
print("스모크와 본 학습: 동일한 ResNet-50 FPN·FrozenBatchNorm·640 크기 제한")


**결과 확인:** 실제 batch 16의 GPU 메모리 판정에는 Colab CUDA GPU가 필요하다.
2.3에서는 forward·backward뿐 아니라 AdamW의 첫 step으로 optimizer 상태 메모리까지 확인한다.


### 2.3 실제 batch 16·640 입력의 스모크 학습

**목적:** 본 학습과 같은 구조·입력 크기·AdamW·FP32로 16장을 한 번에 처리할 수 있는지 확인한다.
16장을 작은 배치로 나눠 누적하지 않는다. loss·역전파·첫 optimizer step·가중치 변경과 GPU 최대 메모리를 출력한다.
메모리가 부족하면 명확히 중단하며 batch를 자동으로 줄이지 않는다.


In [ ]:
import gc


def run_smoke_training_step(model, loader):
    """실제 배치 16의 학습 한 번과 optimizer 상태 메모리를 확인한다.

    Args:
        model (torch.nn.Module): 본 학습과 같은 구조의 모델.
        loader (DataLoader): 16장을 반환하는 train 로더.

    Returns:
        dict: 통과 여부·loss·실제 배치 수·GPU 메모리 사용량.
    """
    if (tuple(model.transform.min_size) != (BASELINE_MIN_SIZE,)
            or model.transform.max_size != BASELINE_MAX_SIZE):
        raise RuntimeError("모델 입력 크기가 2.1 설정과 다릅니다. 2.2에서 모델을 다시 생성하세요.")
    images_device, targets_device = [], []
    loss_dict = total_loss = None
    optimizer = create_training_optimizer(model)
    first_parameter = next(p for p in model.parameters() if p.requires_grad)
    parameter_before = first_parameter.detach().clone()
    transformed = {}

    def capture_input_shape(module, inputs, outputs):
        transformed["batch_tensor_shape"] = tuple(outputs[0].tensors.shape)
        transformed["image_sizes"] = [list(size) for size in outputs[0].image_sizes]
        transformed["input_dtype"] = str(outputs[0].tensors.dtype)

    handle = model.transform.register_forward_hook(capture_input_shape)
    cuda = DEVICE.type == "cuda"
    if cuda:
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    start = time.perf_counter()
    report = {"passed": False, "configuration": smoke_configuration()}
    try:
        model.train()
        images, targets = next(iter(loader))
        if len(images) != BASELINE_BATCH_SIZE:
            raise RuntimeError(f"스모크 입력이 {len(images)}장입니다. 2.1에서 batch 16 로더를 다시 구성하세요.")
        images_device = [image.to(DEVICE, non_blocking=PIN_MEMORY) for image in images]
        targets_device = [{key: value.to(DEVICE, non_blocking=PIN_MEMORY) for key, value in target.items()} for target in targets]
        loss_dict = model(images_device, targets_device)
        total_loss = sum(loss_dict.values())
        if not torch.isfinite(total_loss):
            raise FloatingPointError("스모크 loss가 유한하지 않습니다. 2.1의 입력·bbox와 학습 설정을 확인하세요.")
        optimizer.zero_grad(set_to_none=True)
        total_loss.backward()
        optimizer.step()
        if cuda:
            torch.cuda.synchronize()
        changed = not torch.equal(parameter_before, first_parameter.detach())
        if not changed:
            raise RuntimeError("가중치가 업데이트되지 않았습니다. 2.2와 2.3을 순서대로 다시 확인하세요.")
        report.update({
            "passed": True, "actual_batch_size": len(images),
            "loss": float(total_loss.detach().cpu()),
            "loss_items": {key: float(value.detach().cpu()) for key, value in loss_dict.items()},
            "optimizer": type(optimizer).__name__, "optimizer_state_entries": len(optimizer.state),
            "parameter_changed": changed, **transformed,
        })
    except torch.OutOfMemoryError:
        report["error"] = "GPU 메모리 부족: 실제 batch 16·640·FP32의 첫 AdamW step을 통과하지 못했습니다."
    finally:
        handle.remove()
        report["seconds"] = time.perf_counter() - start
        if cuda:
            gib = 1024 ** 3
            report["gpu_total_gb"] = torch.cuda.get_device_properties(DEVICE).total_memory / gib
            report["peak_allocated_gb"] = torch.cuda.max_memory_allocated() / gib
            report["peak_reserved_gb"] = torch.cuda.max_memory_reserved() / gib
        model.zero_grad(set_to_none=True)
        del optimizer, parameter_before, images_device, targets_device, loss_dict, total_loss
        gc.collect()
        if cuda:
            torch.cuda.empty_cache()
    return report


SMOKE_BATCH16_PASSED = False
SMOKE_VALIDATED_CONFIGURATION = None
if DEVICE.type != "cuda":
    raise RuntimeError("GPU batch 16 검사에는 CUDA가 필요합니다. Colab GPU 커널로 연결하고 0.3부터 확인하세요.")
smoke_report = run_smoke_training_step(model, train_loader)
if not smoke_report["passed"]:
    print(smoke_report.get("error", "스모크 테스트 실패"), flush=True)
    print(f"GPU 최대 사용: {smoke_report.get('peak_allocated_gb', 0):.2f} GiB / 총 {smoke_report.get('gpu_total_gb', 0):.2f} GiB", flush=True)
    raise RuntimeError("GPU 메모리를 확인하세요. GPU 또는 공통 batch 조건을 조정한 뒤 2.1~2.3을 다시 실행하세요. batch를 자동으로 줄이지 않습니다.")
SMOKE_BATCH16_PASSED = True
SMOKE_VALIDATED_CONFIGURATION = smoke_report["configuration"]
print(f"실제 batch: {smoke_report['actual_batch_size']}장 | 모델 내부 배치 텐서: {smoke_report['batch_tensor_shape']}")
print("optimizer:", smoke_report["optimizer"], "| optimizer 상태 생성:", smoke_report["optimizer_state_entries"])
print("loss:", smoke_report["loss_items"])
print(f"GPU 최대 사용: {smoke_report['peak_allocated_gb']:.2f} GiB / 총 {smoke_report['gpu_total_gb']:.2f} GiB")
print(f"GPU 최대 예약: {smoke_report['peak_reserved_gb']:.2f} GiB | 시간: {smoke_report['seconds']:.1f}초")
print("batch 16·640 입력·FP32·AdamW step·가중치 업데이트: 통과")


**결과 확인:** 실제 batch 16, 모델 내부 긴 변 제한 640, AdamW 상태 생성, 가중치 업데이트가 모두 확인돼야 한다.
이 결과는 첫 배치의 메모리 확인이며 전체 학습의 모든 이미지 조합에서 메모리 부족이 없다는 보장은 아니다.
스모크 테스트의 점수나 한 배치 loss로 성능을 판단하지 않는다. 통과 후 2.4에서 검증 추론도 확인한다.


### 2.4 검증 이미지 예측·시각화

**목적:** val 이미지에서 모델 예측을 받아 bbox 시각화까지 연결되는지 확인한다.

초록색 bbox는 정답, 빨간색 bbox는 모델 예측이다. 스모크 모델은 무작위 초기화 상태라 예측 점수가 낮을 수 있다. 시각화 흐름 확인을 위해 추론 단계에서는 낮은 점수 후보도 받아 이미지마다 상위 5개까지 표시한다. 빨간색 박스가 있다는 사실은 정답을 맞혔다는 뜻이 아니다.


In [ ]:
# 기본 score threshold에서 모두 제거될 수 있으므로, smoke 시각화에서는 낮은 점수 후보도 받는다.
# 이는 시각화 점검용 설정이며 본 학습의 예측 임계값으로 사용하지 않는다.
model.roi_heads.score_thresh = 0.0
val_predictions = []
model.eval()
with torch.inference_mode():
    for images, targets in val_loader:
        image = images[0].to(DEVICE)
        prediction = model([image])[0]
        val_predictions.append({
            "image": image.cpu(),
            "target": {key: value.cpu() for key, value in targets[0].items()},
            "prediction": {key: value.detach().cpu() for key, value in prediction.items()},
        })
        if len(val_predictions) >= VAL_PREVIEW_COUNT:
            break

assert val_predictions, "검증 이미지를 추론하지 못했습니다."
print(f"검증 추론 완료: {len(val_predictions)}장")
print("이미지별 추론 후보 수:", [len(item["prediction"]["boxes"]) for item in val_predictions])

# 정답/예측 bbox를 같은 이미지에 그려 육안 확인한다.
fig, axes = plt.subplots(len(val_predictions), 1, figsize=(9, 5 * len(val_predictions)))
if len(val_predictions) == 1:
    axes = [axes]
for preview_index, (ax, item) in enumerate(zip(axes, val_predictions)):
    draw_target = item["target"]
    draw_prediction = item["prediction"]
    top_indices = torch.argsort(draw_prediction["scores"], descending=True)[:5]
    file_name = val_frame.iloc[preview_index]["file_name"]
    canvas = draw_boxes(
        TRAIN_IMAGE_DIR / file_name,
        [xyxy_to_xywh(box.tolist()) for box in draw_target["boxes"]],
        [f"GT:{model_label_to_category[int(label)]}" for label in draw_target["labels"]],
        color="lime", width=4,
    )
    canvas = draw_boxes(
        canvas,
        [xyxy_to_xywh(draw_prediction["boxes"][index].tolist()) for index in top_indices],
        [f"Pred:{model_label_to_category[int(draw_prediction['labels'][index])]} "
         f"{float(draw_prediction['scores'][index]):.2f}" for index in top_indices],
        color="red", width=3,
    )
    ax.imshow(canvas)

    ax.set_title(f"green=GT / red=prediction | {len(draw_target['boxes'])} GT, "
                 f"top {len(top_indices)} prediction candidates")
    ax.axis("off")
plt.tight_layout()
plt.show()

print("검증 이미지 시각화 완료:", len(val_predictions), "장")


**결과 확인:** 초록색 정답 bbox와 빨간색 상위 예측 후보가 함께 표시되는지 확인한다. 빨간색 후보는 낮은 점수일 수 있으며, 이 시각화는 모델이 맞혔는지 평가하는 용도가 아니다. 스모크 모델의 결과로 성능 결론을 내리지 않는다.

**스모크 테스트 통과 기준:** 실제 batch 16·640 크기 제한·FP32·AdamW의 forward/backward/step, 유한한 loss, 가중치 업데이트, 검증 추론 및 bbox 시각화가 오류 없이 끝난다. W&B는 이 단계를 통과한 뒤 본 학습 전에 연결한다. 혼동행렬은 첫 베이스라인을 여러 시드로 학습한 뒤 동일한 val 분할과 평가 기준으로 비교하고, 주 성능 지표 mAP@[0.75:0.95]을 보완하는 분석으로 사용한다.


## 3. 사전학습 조건별 seed 0·1·2 학습

**목적:** 사전학습 없는 베이스라인의 학습·제출을 먼저 마친 뒤, COCO 사전학습을 추가 실험한다.
분할 seed 410, train 185장 / val 47장, 수정된 771개 bbox는 고정한다.

| 순서 | 실행 위치 | 진행 내용 |
|---|---|---|
| 1 | 3.1~3.4 | 공통 설정·W&B·학습 함수 준비 |
| 2 | 3.5.1 → 3.5.2 → 3.5.3 | 사전학습 없음 seed 0 → 1 → 2 학습 |
| 3 | 3.6 → 4번 | 사전학습 없음 결과 확인 → seed별 CSV 3개 생성·Kaggle 제출 |
| 4 | 3.7.1 → 3.7.2 → 3.7.3 | 위 3건 제출 후 돌아와 COCO seed 0 → 1 → 2 학습 |
| 5 | 3.8 → 4번 | COCO 결과 확인 → seed별 CSV 3개 생성·Kaggle 제출 |
| 6 | 3.9 | 두 조건의 seed별·평균 validation 결과 비교 |

**총 6회 학습·6회 제출**이다. 중간에 한 조건을 선택해 다른 조건의 seed를 생략하지 않는다.
각 조건·seed마다 새 모델로 학습하며, 이전 seed의 학습 가중치를 이어받지 않는다.
최고 validation mAP@[0.75:0.95]의 `best.pt`를 각 seed의 Test 예측에 사용한다.
W&B와 Drive에는 초기화 조건과 seed를 함께 기록해 결과를 구분한다.


### 3.1 학습 전 환경 재확인

**목적:** 0번에서 맞춘 잠금 버전과 GPU를 다시 확인하고 스모크 모델의 GPU 메모리를 비운다.
패키지는 여기서 추가 설치하지 않는다. 새 패키지가 필요하면 README에 따라 `uv add`로 관리한다.


In [ ]:
RUNTIME_ENVIRONMENT = validate_training_environment()
if (not globals().get("SMOKE_BATCH16_PASSED", False)
        or globals().get("SMOKE_VALIDATED_CONFIGURATION") != smoke_configuration()):
    raise RuntimeError("현재 설정의 batch 16 스모크 검사가 필요합니다. 2.1~2.4를 다시 실행한 뒤 3.1을 실행하세요.")
if "IMAGE_CONTENT_SHA256" not in globals():
    raise RuntimeError("로컬 이미지 캐시가 준비되지 않았습니다. 1.7과 2.1을 실행한 뒤 3.1을 다시 실행하세요.")
for current_dataset in (train_dataset, val_dataset):
    if current_dataset.image_dir.resolve() != TRAIN_IMAGE_DIR.resolve():
        raise RuntimeError("Dataset이 이전 이미지 경로를 사용합니다. 2.1을 다시 실행해 로컬 경로로 연결하세요.")

import wandb
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# 스모크 테스트의 모델·optimizer·GPU 텐서는 본 학습에 재사용하지 않는다.
for variable_name in ("model", "optimizer", "first_parameter", "parameter_before",
                      "images", "targets", "image", "loss_dict", "total_loss"):
    globals().pop(variable_name, None)
torch.cuda.empty_cache()
print("팀 잠금 버전 · GPU 확인 완료. 본 학습은 seed마다 새 모델로 시작합니다.")


### 3.2 W&B 인증

**목적:** Colab에서 팀 W&B 프로젝트에 기록할 권한을 확인한다.
`WANDB_API_KEY` 환경변수가 없으면 숨김 입력으로 받고, 키를 셀·출력·설정 파일에 저장하지 않는다.
`[1/2] API 키 입력 대기`이면 VSCode 상단 입력창에 키를 넣고 Enter를 누른다.
`[2/2] W&B 서버 인증 중`이면 입력은 끝났고 서버 응답을 기다리는 상태다.
입력 대기 중이라는 이유로 셀을 중단하지 않는다. VSCode·Colab 연결에 따라 중단 후 커널 재시작이 요구될 수 있다.


In [ ]:
import getpass

wandb_api_key = None
try:
    wandb_api_key = os.environ.get("WANDB_API_KEY")
    if not wandb_api_key:
        print("[1/2] API 키 입력 대기: VSCode 화면 맨 위의 숨김 입력창에 키를 넣고 Enter를 누르세요.", flush=True)
        wandb_api_key = getpass.getpass("W&B API key (숨김 입력): ")
    if not wandb_api_key:
        raise ValueError("W&B API key가 비어 있습니다. 3.2 셀을 다시 실행해 숨김 입력하세요.")
    print("[2/2] 키 입력 완료. W&B 서버 인증 중…", flush=True)
    if not wandb.login(key=wandb_api_key, relogin=True, verify=True):
        raise RuntimeError("W&B 인증이 완료되지 않았습니다. 계정·키·Colab 인터넷 연결을 확인하세요.")
finally:
    del wandb_api_key

WANDB_ENTITY = os.environ.get("WANDB_ENTITY") or "codeit-prj1-team2"
WANDB_PROJECT = "pill-object-detection"
print(f"W&B 인증 완료 | {WANDB_ENTITY}/{WANDB_PROJECT}", flush=True)


### 3.3 두 실험의 공통 설정

| 설정 | A: `scratch` | B: `coco` |
|---|---|---|
| 모델 | Faster R-CNN ResNet-50 FPN | 동일 |
| 초기 가중치 | 무작위, ImageNet도 사용하지 않음 | COCO_V1 검출 가중치 |
| 약 분류·회귀 헤드 | 약 56종＋배경, 해당 학습 seed로 새로 초기화 | 같은 구조·seed로 새로 초기화 |
| 학습 범위 | 모든 학습 파라미터 | 동일, 백본 일부 고정 안 함 |
| 정규화 | FrozenBatchNorm2d | 동일 |
| 분할 / 학습 seed | 분할 410 / 학습 0·1·2 각각 | 동일 |
| 최대 epoch / 조기 종료 | 100 / 최소 20, patience 30, min_delta 0.001 | 동일 |
| batch | 16 | 동일 |
| optimizer | AdamW 명시 선택 | 동일 |
| AdamW 초기값 | lr 0.000167(56종), betas 0.9·0.999, weight decay 0.0005 | 동일 |
| decay 제외 | bias·정규화의 1차원 파라미터 | 동일 |
| 입력 크기 | 종횡비 유지, 긴 변 최대 640 (`min_size=640`, `max_size=640`) | 동일 |
| 증강 | torchvision 모델 기본값: 추가 랜덤 증강 없음 | 동일 |
| 이미지 읽기 | 1.7에서 검증한 Colab 로컬 캐시 | 동일 |
| 데이터 로딩 작업자 | 0 (노트북 안정성 우선) | 동일 |
| 수치 정밀도 | FP32, AMP 사용 안 함 | 동일 |
| W&B Media | 매 epoch, 같은 검증 이미지 3장 | 동일 |

**YOLO 설정을 옮긴 범위와 차이:**
- `epochs=100`, `patience=30`, 실제 학습 batch 16, workers 0을 맞춘다. 요청에 따라 AMP는 사용하지 않는다.
- torchvision에는 optimizer auto 선택기가 없으므로 AdamW를 직접 사용한다. 현재 잠금 버전 Ultralytics의 auto AdamW 식 `round(0.002*5/(4+클래스 수), 6)`으로 초기 lr을 정한다. 56종이면 0.000167이며 실제 값은 2.1·W&B에 기록한다.
- YOLO의 `imgsz=640`은 보통 정사각형 letterbox, 여기서는 Faster R-CNN 내부의 종횡비 유지 리사이즈·배치 패딩을 사용한다. 같은 숫자여도 픽셀 배치 방식은 완전히 같지 않다.
- YOLO 기본 증강에는 Mosaic·HSV·뒤집기 등이 포함되지만 torchvision 검출 모델 자체는 자동 증강을 제공하지 않는다. 따라서 ‘각 라이브러리 기본값’은 같은 증강이 아니다. 이번 비교는 모델과 기본 학습 파이프라인의 비교로 기록한다.
- YOLO 기본 warmup·LR 스케줄러·gradient accumulation까지 복제하지 않는다. Faster R-CNN은 고정 lr, accumulation 1이다. 동일 optimizer 이름만으로 전체 학습법이 같아지는 것은 아니다.

[YOLO 학습 설정](https://docs.ultralytics.com/modes/train/#train-settings),
[torchvision 미세조정 예제](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html)

**통제하는 이유:** torchvision은 가중치 유무에 따라 BatchNorm 종류와 백본 고정 기본값도 달라진다.
두 실험은 정규화 구조를 COCO의 FrozenBatchNorm으로 통일하고, 모든 학습 파라미터를 업데이트한다.
정규화의 통계·scale·bias는 두 조건 모두 학습 중 고정된다. scratch는 초기값, COCO는 사전학습 값에서 시작한다.
스모크도 같은 FrozenBatchNorm 모델을 사용한다. 이전 일반 BatchNorm·SGD scratch 결과를 이번 비교에 섞지 않는다.
[torchvision 구현](https://docs.pytorch.org/vision/stable/_modules/torchvision/models/detection/faster_rcnn.html),
[FrozenBatchNorm 설명](https://docs.pytorch.org/vision/stable/generated/torchvision.ops.FrozenBatchNorm2d.html)

설정값은 **2.1에서 한 번 정의**하고 스모크와 본 학습이 함께 사용한다.
스모크도 본 학습과 같은 실제 batch 16·640 제한·AdamW·FP32로 실행한다. validation 추론 batch는 1이다.
batch 16의 실행 가능 여부는 실제 GPU 메모리에 따라 달라진다. 메모리 부족 오류가 나면 공통 실험 조건을 확인한 뒤 조정하며 코드가 자동으로 batch를 줄이지 않는다.

같은 최대 epoch와 조기 종료 규칙에서의 비교다. 각 조건의 최적 성능을 보장하는 튜닝 실험은 아니다.
COCO 가중치는 첫 실행 시 인터넷으로 내려받아 Colab 캐시에 저장한다. VM이 바뀌면 다시 받을 수 있다.
`best.pt`는 최고 validation mAP가 갱신될 때, `last.pt`는 5 epoch마다와 정상 종료 시 Drive에 저장한다.
체크포인트에는 epoch·모델·optimizer·설정·validation mAP가 포함된다. 임시 파일 저장을 마친 뒤 기존 파일을 교체한다.
저장 파일은 학습 실행 후 생성된다. 자동 학습 재개 기능은 아직 없으며, VM이 강제 종료되면 마지막 저장 이후 최대 5 epoch 미만은 남지 않을 수 있다.
노트북의 다중 프로세스 로딩은 시작 방식·공유 메모리에 따라 멈추거나 실패할 수 있어, 현재 `num_workers=0`을 유지한다.
이미지는 로컬에서 읽고 GPU에서는 pinned memory를 사용해 전송한다. AMP는 적용하지 않는다.
[PyTorch 데이터 로딩 안내](https://docs.pytorch.org/docs/2.14/data.html#single-and-multi-process-data-loading)
epoch 시간은 학습＋validation, 추론 시간은 모델 추론만 측정한다. 가중치 다운로드·모델 생성은 학습 시간에서 제외한다.
`media_seconds`는 bbox 그리기·표시용 이미지 생성·로그 호출 시간이다. 총 학습 시간에는 반복문의 Media 기록 시간도 포함된다.
W&B 백그라운드 전송은 이 측정 이후에도 진행될 수 있다.


In [ ]:
import csv
import json
import random
import time
from datetime import datetime

from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import FrozenBatchNorm2d

# 실제 설정값은 2.1에서 정의하고 2.3에서 검증한 값을 그대로 사용한다.
if (not globals().get("SMOKE_BATCH16_PASSED", False)
        or globals().get("SMOKE_VALIDATED_CONFIGURATION") != smoke_configuration()):
    raise RuntimeError("스모크 검사 후 설정이 변경됐습니다. 2.1~2.4부터 다시 실행하세요.")
ENVIRONMENT_ID = RUNTIME_ENVIRONMENT["uv_lock_sha256"][:12]
EXPERIMENT_RECIPE = {
    "recipe_version": "pretraining_adamw640_batch16_v2",
    "model": "fasterrcnn_resnet50_fpn",
    "normalization": "FrozenBatchNorm2d_eps0_both_conditions",
    "trainable_scope": "all_parameters",
    "environment": RUNTIME_ENVIRONMENT,
    "split_sha256": hashlib.sha256(SPLIT_PATH.read_bytes()).hexdigest(),
    "label_fixes_sha256": hashlib.sha256(FIXES_PATH.read_bytes()).hexdigest(),
    "class_mapping": {str(k): int(v) for k, v in category_to_model_label.items()},
    "max_epochs": MAX_EPOCHS,
    "early_stopping": [EARLY_STOP_MIN_EPOCHS, EARLY_STOP_PATIENCE, EARLY_STOP_MIN_DELTA],
    "batch_size": BASELINE_BATCH_SIZE,
    "optimizer": BASELINE_OPTIMIZER,
    "lr": BASELINE_LR,
    "betas": list(BASELINE_BETAS),
    "weight_decay": BASELINE_WEIGHT_DECAY,
    "image_size": [BASELINE_MIN_SIZE, BASELINE_MAX_SIZE],
    "score_threshold": BASELINE_SCORE_THRESHOLD,
    "map_ious": MAP_IOU_THRESHOLDS,
    "augmentation": "torchvision_detector_default_no_random_augmentation",
    "resize_policy": "aspect_ratio_preserved_long_side_at_most_640",
    "lr_source": "ultralytics_locked_auto_AdamW_class_count_formula",
    "lr_scheduler": "none",
    "gradient_accumulation_steps": 1,
    "image_content_sha256": IMAGE_CONTENT_SHA256,
    "image_storage": "colab_local_cache",
    "num_workers": DATA_LOADER_WORKERS,
    "pin_memory": PIN_MEMORY,
    "precision": "fp32",
    "amp": USE_AMP,
    "media": {
        "every_epochs": 1,
        "image_count": WANDB_PREVIEW_COUNT,
        "score_threshold": WANDB_PREVIEW_SCORE_THRESHOLD,
        "preview_max_side": WANDB_PREVIEW_MAX_SIDE,
        "file_type": "jpg",
        "fixed_files": val_frame["file_name"].head(WANDB_PREVIEW_COUNT).tolist(),
    },
}
EXPERIMENT_ID = hashlib.sha256(
    json.dumps(EXPERIMENT_RECIPE, sort_keys=True).encode()
).hexdigest()[:12]
EXPERIMENT_SUMMARY_PATH = RUNS_ROOT / f"pretraining_comparison_{EXPERIMENT_ID}.csv"

print("실험 순서: scratch/seed 0·1·2 → Kaggle 3건 제출 → coco/seed 0·1·2 → Kaggle 3건 제출")
print("본 학습 설정: 최대", MAX_EPOCHS, "epoch | batch", BASELINE_BATCH_SIZE)
print("고정 분할 seed:", SPLIT_SEED)
print("평가 지표: mAP@[0.75:0.95]")
print("실험별 비교표:", EXPERIMENT_SUMMARY_PATH)
print("이미지: Colab 로컬 | 작업자:", DATA_LOADER_WORKERS, "| AMP:", USE_AMP)
print("폴더 이름 예: scratch_s0_1028_1530 / coco_s0_1028_1530")


### 3.4 공통 모델 생성·학습·결과 기록 함수

2.2에서 정의한 같은 모델 생성·AdamW 함수를 사용하고, 초기화 조건만 바꿔 학습한다. 새 약 분류·회귀 헤드는 두 조건에 같은 seed로 만든다.
W&B run 이름·태그에 `scratch` 또는 `coco`와 seed를 표시하고, 같은 실험 ID로 묶는다.
폴더 이름은 `scratch_s0_1028_1530`처럼 조건·seed·월일·시간만 쓴다.
같은 분에 같은 조건을 실행하면 `_2` 등을 붙여 기존 결과를 덮어쓰지 않는다. W&B 이름도 같은 이름을 사용한다.
모델 종류·최대 epoch·전체 시작 날짜 등 상세 정보는 config.json과 W&B에 기록한다.
각 run에 `config.json`, `environment.json`, `class_map.csv`, `metrics.csv`, `result.json`, `best.pt`, `last.pt`를 저장한다.
환경·분할·수정 라벨·학습 설정이 같은 결과만 비교표에 누적하고, 이미 완료된 조건·seed는 재사용한다.
**매 epoch 종료 후** 같은 validation 이미지 3장의 정답(초록)·예측(빨강)을 `Media/val_predictions`에 기록한다.
W&B run의 **Media → val_predictions → Step 슬라이더**에서 epoch 1, 2, 3…의 변화를 확인한다.
Step은 batch 번호가 아니라 **완료한 epoch 번호**다. 각 Step에는 원본 이미지 기준 3장이 들어가며, batch 묶음 3개가 아니다.
이미지마다 파일명·epoch·정답/예측 수·표시 임계값을 caption에 남긴다. score 0.30 이상 예측이 없으면 초록 정답만 보일 수 있다.
검증 추론 결과를 재사용하므로 이미지를 위해 모델 추론을 추가하지 않는다. 표시용 복사본만 긴 변 최대 1280의 JPEG로 기록한다.
Media는 epoch가 끝난 뒤 나타나며 서버 전송·화면 반영에 시간이 조금 걸릴 수 있다.
[W&B 이미지와 Step 슬라이더 안내](https://docs.coreweave.com/products/wandb/track/log/media#images)


In [ ]:
def set_training_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def save_checkpoint(path, epoch, model, optimizer, val_map, config):
    """임시 파일 저장을 마친 뒤 best/last 체크포인트를 교체한다.

    Args:
        path (Path): Drive의 best.pt 또는 last.pt 저장 경로.
        epoch (int): 저장하는 완료 epoch.
        model (torch.nn.Module): 학습 모델.
        optimizer (torch.optim.Optimizer): optimizer.
        val_map (float): 해당 epoch의 validation mAP.
        config (dict): 모델·환경·학습 설정.
    """
    path = Path(path)
    temporary_path = path.with_name(path.name + ".tmp")
    try:
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_map_75_95": val_map,
            "config": config,
        }, temporary_path)
        os.replace(temporary_path, path)
    finally:
        temporary_path.unlink(missing_ok=True)


def create_run_directory(condition, train_seed):
    """조건·seed·짧은 날짜만으로 겹치지 않는 run 폴더를 만든다.

    Args:
        condition (str): scratch 또는 coco.
        train_seed (int): 학습 seed.

    Returns:
        Path: 새로 생성한 run 폴더.
    """
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    base_name = f"{condition}_s{train_seed}_{datetime.now().strftime('%m%d_%H%M')}"
    sequence = 1
    while True:
        suffix = "" if sequence == 1 else f"_{sequence}"
        run_dir = RUNS_ROOT / (base_name + suffix)
        try:
            run_dir.mkdir(exist_ok=False)
            return run_dir
        except FileExistsError:
            sequence += 1


def evaluate_map_and_speed(model, loader):
    """검증 지표·추론 속도와 고정 이미지의 예측을 한 번에 수집한다.

    Args:
        model (torch.nn.Module): 평가 모델.
        loader (DataLoader): shuffle=False인 validation 로더.

    Returns:
        tuple: validation mAP, 이미지당 추론 ms, W&B 표시용 CPU 예측 목록.
    """
    metric = MeanAveragePrecision(
        box_format="xyxy", iou_type="bbox", iou_thresholds=MAP_IOU_THRESHOLDS,
    )
    model.eval()
    inference_seconds, image_count = 0.0, 0
    preview_records = []
    with torch.inference_mode():
        for images, targets in loader:
            images_device = [image.to(DEVICE, non_blocking=PIN_MEMORY) for image in images]
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            outputs = model(images_device)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            inference_seconds += time.perf_counter() - start
            cpu_outputs = [{key: value.detach().cpu() for key, value in output.items()} for output in outputs]
            cpu_targets = [{key: value.detach().cpu() for key, value in target.items()} for target in targets]
            metric.update(cpu_outputs, cpu_targets)
            for offset, (prediction, target) in enumerate(zip(cpu_outputs, cpu_targets)):
                if len(preview_records) >= WANDB_PREVIEW_COUNT:
                    break
                file_name = loader.dataset.frame.iloc[image_count + offset]["file_name"]
                preview_records.append({"file_name": file_name, "target": target, "prediction": prediction})
            image_count += len(images)
    result = metric.compute()
    return float(result["map"].item()), 1000 * inference_seconds / max(image_count, 1), preview_records


def make_wandb_preview(preview_records, epoch):
    """검증 결과를 재사용해 같은 이미지의 에폭별 정답·예측 bbox를 기록한다.

    Args:
        preview_records (list[dict]): 고정 validation 이미지의 CPU 정답·예측.
        epoch (int): 완료한 epoch. W&B Step과 같은 값.

    Returns:
        list: W&B에 기록할 이미지 목록.
    """
    previews = []
    for record in preview_records:
        target, prediction = record["target"], record["prediction"]
        file_name = record["file_name"]
        canvas = draw_boxes(
            TRAIN_IMAGE_DIR / file_name,
            [xyxy_to_xywh(box.tolist()) for box in target["boxes"]],
            [f"GT:{model_label_to_category[int(label)]}" for label in target["labels"]],
            color="lime", width=4,
        )
        eligible = torch.where(prediction["scores"] >= WANDB_PREVIEW_SCORE_THRESHOLD)[0]
        ranked = torch.argsort(prediction["scores"][eligible], descending=True)
        indices = eligible[ranked][:20].tolist()
        canvas = draw_boxes(
            canvas,
            [xyxy_to_xywh(prediction["boxes"][index].tolist()) for index in indices],
            [f"Pred:{model_label_to_category[int(prediction['labels'][index])]} "
             f"{float(prediction['scores'][index]):.2f}" for index in indices],
            color="red", width=3,
        )
        # W&B용 복사본만 줄이며 원본 이미지와 모델 입력·평가는 그대로 둔다.
        canvas.thumbnail((WANDB_PREVIEW_MAX_SIDE, WANDB_PREVIEW_MAX_SIDE))
        caption = (
            f"epoch {epoch} | {file_name} | green=GT:{len(target['boxes'])}, "
            f"red=Pred:{len(indices)} (score>={WANDB_PREVIEW_SCORE_THRESHOLD:.2f}, max20)"
        )
        previews.append(wandb.Image(canvas, caption=caption, file_type="jpg"))
    return previews


def run_training_seed(train_seed, condition):
    if train_seed not in TRAIN_SEEDS:
        raise ValueError(f"허용된 학습 seed가 아닙니다: {train_seed}")

    if condition not in EXPERIMENT_CONDITIONS:
        raise ValueError("초기화 조건은 scratch 또는 coco입니다. 3.3 설정을 확인하세요.")
    environment = validate_training_environment()
    if environment != RUNTIME_ENVIRONMENT:
        raise RuntimeError("실행 환경이 달라졌습니다. 3.1부터 다시 실행해 비교표를 분리하세요.")
    set_training_seed(train_seed)
    run_dir = create_run_directory(condition, train_seed)
    run_name = run_dir.name
    metrics_path = run_dir / "metrics.csv"
    config_path = run_dir / "config.json"

    config = {
        "model": "torchvision_fasterrcnn_resnet50_fpn",
        "started_at": datetime.now().isoformat(),
        "run_name": run_name,
        "image_source": str(DRIVE_TRAIN_IMAGE_DIR),
        "image_read_dir": str(TRAIN_IMAGE_DIR),
        "image_content_sha256": IMAGE_CONTENT_SHA256,
        "num_workers": DATA_LOADER_WORKERS,
        "pin_memory": PIN_MEMORY,
        "precision": "fp32",
        "amp": False,
        "checkpoint_every_epochs": CHECKPOINT_EVERY_EPOCHS,
        "media": EXPERIMENT_RECIPE["media"],
        "pretrained_weights": condition == "coco",
        "initialization": condition,
        "weights_enum": EXPERIMENT_CONDITIONS[condition]["weights"],
        "trainable_scope": "all_parameters",
        "normalization": EXPERIMENT_RECIPE["normalization"],
        "experiment_id": EXPERIMENT_ID,
        "experiment_recipe": EXPERIMENT_RECIPE,
        "split_seed": int(SPLIT_SEED),
        "train_seed": int(train_seed),
        "max_epochs": MAX_EPOCHS,
        "early_stopping": {
            "min_epochs": EARLY_STOP_MIN_EPOCHS,
            "patience": EARLY_STOP_PATIENCE,
            "min_delta": EARLY_STOP_MIN_DELTA,
            "monitor": "val_map_75_95",
        },
        "batch_size": BASELINE_BATCH_SIZE,
        "optimizer": BASELINE_OPTIMIZER,
        "learning_rate": BASELINE_LR,
        "lr_source": "ultralytics_locked_auto_AdamW_class_count_formula",
        "lr_scheduler": "none",
        "gradient_accumulation_steps": 1,
        "augmentation": "torchvision_detector_default_no_random_augmentation",
        "resize_policy": "aspect_ratio_preserved_long_side_at_most_640",
        "smoke_batch16_configuration": SMOKE_VALIDATED_CONFIGURATION,
        "betas": list(BASELINE_BETAS),
        "weight_decay": BASELINE_WEIGHT_DECAY,
        "min_size": BASELINE_MIN_SIZE,
        "max_size": BASELINE_MAX_SIZE,
        "score_threshold": BASELINE_SCORE_THRESHOLD,
        "num_classes_including_background": NUM_CLASSES,
        "map_iou_thresholds": MAP_IOU_THRESHOLDS,
        "train_images": len(train_dataset),
        "val_images": len(val_dataset),
        "fixed_boxes": len(fixed_annotations),
        "environment": environment,
        "category_to_model_label": {str(k): int(v) for k, v in category_to_model_label.items()},
        "split_sha256": hashlib.sha256(SPLIT_PATH.read_bytes()).hexdigest(),
        "label_fixes_sha256": hashlib.sha256(FIXES_PATH.read_bytes()).hexdigest(),
    }
    config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    (run_dir / "environment.json").write_text(json.dumps(environment, ensure_ascii=False, indent=2), encoding="utf-8")
    class_map.to_csv(run_dir / "class_map.csv", index=False)

    generator = torch.Generator().manual_seed(train_seed)
    train_loader_for_run = DataLoader(
        train_dataset,
        batch_size=BASELINE_BATCH_SIZE,
        shuffle=True,
        num_workers=DATA_LOADER_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=detection_collate_fn,
        generator=generator,
    )
    val_loader_for_run = DataLoader(
        val_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=DATA_LOADER_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=detection_collate_fn,
    )

    model = build_experiment_model(condition, train_seed)
    optimizer = create_training_optimizer(model)

    fieldnames = [
        "epoch", "train_loss", "val_map_75_95", "epoch_seconds",
        "val_inference_ms_per_image", "media_seconds",
    ]
    best_map_75_95 = -1.0
    best_epoch = 0
    best_for_stopping = -1.0
    epochs_without_improvement = 0
    last_epoch = 0
    total_training_seconds = 0.0
    total_media_seconds = 0.0
    final_val_map = -1.0

    wandb_run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        dir=str(run_dir),
        name=run_name,
        config=config,
        group=f"pretraining-{EXPERIMENT_ID}",
        tags=["faster-rcnn", "baseline", condition, f"seed-{train_seed}"],
    )
    wandb_run.define_metric("epoch")
    wandb_run.define_metric("train/*", step_metric="epoch")
    wandb_run.define_metric("val/*", step_metric="epoch")
    wandb_run.define_metric("time/*", step_metric="epoch")
    wandb_run.define_metric("Media/*", step_metric="epoch")

    training_start = time.perf_counter()
    try:
        with metrics_path.open("w", newline="", encoding="utf-8") as metrics_file:
            metric_writer = csv.DictWriter(metrics_file, fieldnames=fieldnames)
            metric_writer.writeheader()
            for epoch in range(1, MAX_EPOCHS + 1):
                epoch_start = time.perf_counter()
                model.train()
                running_loss = 0.0
                batch_count = 0

                for images, targets in train_loader_for_run:
                    images = [image.to(DEVICE, non_blocking=PIN_MEMORY) for image in images]
                    targets = [
                        {key: value.to(DEVICE, non_blocking=PIN_MEMORY) for key, value in target.items()}
                        for target in targets
                    ]
                    loss_dict = model(images, targets)
                    total_loss = sum(loss for loss in loss_dict.values())
                    if not torch.isfinite(total_loss):
                        raise FloatingPointError(
                            f"seed {train_seed}, epoch {epoch}에서 유한하지 않은 loss: {loss_dict}"
                        )
                    optimizer.zero_grad(set_to_none=True)
                    total_loss.backward()
                    optimizer.step()
                    running_loss += float(total_loss.detach().cpu())
                    batch_count += 1

                train_loss = running_loss / max(batch_count, 1)
                final_val_map, inference_ms, preview_records = evaluate_map_and_speed(model, val_loader_for_run)
                epoch_seconds = time.perf_counter() - epoch_start
                last_epoch = epoch
                if final_val_map > best_map_75_95:
                    best_map_75_95 = final_val_map
                    best_epoch = epoch
                    save_checkpoint(
                        run_dir / "best.pt", epoch, model, optimizer,
                        final_val_map, config,
                    )
                if final_val_map > best_for_stopping + EARLY_STOP_MIN_DELTA:
                    best_for_stopping = final_val_map
                    epochs_without_improvement = 0
                else:
                    epochs_without_improvement += 1
                if epoch % CHECKPOINT_EVERY_EPOCHS == 0:
                    save_checkpoint(
                        run_dir / "last.pt", epoch, model, optimizer,
                        final_val_map, config,
                    )

                # 체크포인트 처리가 끝난 뒤 시각화를 수행해 Media 실패와 가중치 저장을 분리한다.
                media_start = time.perf_counter()
                previews, media_error = [], None
                try:
                    previews = make_wandb_preview(preview_records, epoch)
                except Exception as error:
                    media_error = f"{type(error).__name__}: {error}"
                    print(f"epoch {epoch}: Media 이미지 생성 실패. 학습은 계속합니다. {media_error}", flush=True)
                media_generation_seconds = time.perf_counter() - media_start
                log_payload = {
                    "epoch": epoch,
                    "train/loss": train_loss,
                    "train/lr": optimizer.param_groups[0]["lr"],
                    "val/map_75_95": final_val_map,
                    "time/epoch_seconds": epoch_seconds,
                    "time/val_inference_ms_per_image": inference_ms,
                    "time/media_generation_seconds": media_generation_seconds,
                }
                if previews:
                    log_payload["Media/val_predictions"] = previews
                if media_error:
                    log_payload["Media/preview_error"] = media_error
                # 1 epoch = 1 log = 1 Step. 그래프와 Media가 같은 Step에 기록된다.
                wandb_run.log(log_payload, step=epoch)
                media_seconds = time.perf_counter() - media_start
                total_media_seconds += media_seconds
                row = {
                    "epoch": epoch, "train_loss": train_loss,
                    "val_map_75_95": final_val_map, "epoch_seconds": epoch_seconds,
                    "val_inference_ms_per_image": inference_ms,
                    "media_seconds": media_seconds,
                }
                metric_writer.writerow(row)
                metrics_file.flush()

                print(
                    f"{condition} | seed {train_seed} | epoch {epoch:03d}/{MAX_EPOCHS} "
                    f"| loss={train_loss:.4f} | mAP@[.75:.95]={final_val_map:.4f} "
                    f"| epoch={epoch_seconds:.1f}s | media={media_seconds:.1f}s | val={inference_ms:.1f}ms/image "
                    f"| early-stop={epochs_without_improvement}/{EARLY_STOP_PATIENCE}"
                )
                if (
                    epoch >= EARLY_STOP_MIN_EPOCHS
                    and epochs_without_improvement >= EARLY_STOP_PATIENCE
                ):
                    print(
                        f"Early stopping: {EARLY_STOP_PATIENCE} epoch 동안 "
                        f"mAP 개선 폭이 {EARLY_STOP_MIN_DELTA} 미만입니다."
                    )
                    break

        total_training_seconds = time.perf_counter() - training_start
        # 시각화·업로드 실패가 마지막 학습 가중치 저장을 막지 않게 먼저 저장한다.
        save_checkpoint(run_dir / "last.pt", last_epoch, model, optimizer, final_val_map, config)
        wandb_run.summary["best_val/map_75_95"] = best_map_75_95
        wandb_run.summary["best_epoch"] = best_epoch
        wandb_run.summary["last_epoch"] = last_epoch
        wandb_run.summary["total_training_seconds"] = total_training_seconds
        wandb_run.summary["total_media_seconds"] = total_media_seconds
    finally:
        wandb_run.finish()

    result = {
        "train_seed": train_seed,
        "initialization": condition,
        "experiment_id": EXPERIMENT_ID,
        "environment_id": ENVIRONMENT_ID,
        "status": "complete",
        "best_val_map_75_95": best_map_75_95,
        "best_epoch": best_epoch,
        "last_epoch": last_epoch,
        "total_training_seconds": round(total_training_seconds, 2),
        "total_media_seconds": round(total_media_seconds, 2),
        "best_val_inference_ms_per_image": float(pd.read_csv(metrics_path).set_index("epoch").loc[best_epoch, "val_inference_ms_per_image"]),
        "run_name": run_name,
        "run_dir": str(run_dir),
        "best_checkpoint": str(run_dir / "best.pt"),
    }
    (run_dir / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print("학습 완료:", result)
    return result



SUMMARY_FIELDS = [
    "initialization", "train_seed", "experiment_id", "environment_id", "status",
    "best_val_map_75_95", "best_epoch", "last_epoch", "total_training_seconds", "total_media_seconds",
    "best_val_inference_ms_per_image", "run_name", "run_dir", "best_checkpoint",
]


def load_experiment_results():
    """현재 실험의 완료 결과를 초기화 조건·seed별로 불러온다.

    Returns:
        pandas.DataFrame: 현재 설정으로 완료된 학습 결과표.
    """
    if not EXPERIMENT_SUMMARY_PATH.is_file():
        return pd.DataFrame(columns=SUMMARY_FIELDS)
    results = pd.read_csv(EXPERIMENT_SUMMARY_PATH, dtype={"experiment_id": str, "environment_id": str})
    if not set(SUMMARY_FIELDS).issubset(results.columns):
        raise ValueError("비교표의 열이 다릅니다. 3.3 설정과 비교표 경로를 확인하세요.")
    if not results["experiment_id"].eq(EXPERIMENT_ID).all():
        raise ValueError("다른 설정의 결과가 섞였습니다. 현재 실험의 비교표를 사용하세요.")
    if results.duplicated(["initialization", "train_seed"]).any():
        raise ValueError("같은 조건·seed가 중복됐습니다. 비교표에서 사용할 run을 확인하세요.")
    return results.loc[results["status"].eq("complete")].copy()


def record_experiment_result(result):
    """다른 조건의 seed 0을 덮어쓰지 않고 완료 결과를 저장한다.

    Args:
        result (dict): run_training_seed가 반환한 완료 결과.
    """
    if result["experiment_id"] != EXPERIMENT_ID:
        raise ValueError("현재 실험 설정과 다릅니다. 3.3부터 다시 실행해 결과 경로를 확인하세요.")
    results = load_experiment_results()
    same_run = (
        results["initialization"].eq(result["initialization"])
        & results["train_seed"].eq(result["train_seed"])
    )
    rows = results.loc[~same_run].to_dict("records") + [result]
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    with EXPERIMENT_SUMMARY_PATH.open("w", newline="", encoding="utf-8") as output:
        writer = csv.DictWriter(output, fieldnames=SUMMARY_FIELDS)
        writer.writeheader()
        writer.writerows(rows)
    print("학습 결과 저장:", EXPERIMENT_SUMMARY_PATH)


def get_or_train_result(condition, train_seed):
    """완료된 run은 재사용하고, 없는 조건·seed만 학습한다.

    Args:
        condition (str): scratch 또는 coco.
        train_seed (int): 학습 seed.

    Returns:
        dict: 체크포인트와 validation 성능을 포함한 결과.
    """
    results = load_experiment_results()
    saved = results.loc[
        results["initialization"].eq(condition) & results["train_seed"].eq(train_seed)
    ]
    if not saved.empty:
        result = saved.iloc[0].to_dict()
        if not Path(result["best_checkpoint"]).is_file():
            raise FileNotFoundError("완료 기록의 best.pt가 없습니다. Drive 연결과 해당 run 폴더를 확인하세요.")
        print(f"완료 결과 재사용: {condition} / seed {train_seed} | {result['run_name']}")
        return result
    result = run_training_seed(train_seed, condition)
    record_experiment_result(result)
    return result



def get_condition_results(condition, require_complete=True):
    """조건별 seed 0·1·2 결과와 제출에 사용할 체크포인트를 확인한다.

    Args:
        condition (str): scratch 또는 coco.
        require_complete (bool): 세 seed의 완료를 모두 요구할지 여부.

    Returns:
        pandas.DataFrame: 해당 조건의 seed 순으로 정렬한 완료 결과.
    """
    if condition not in EXPERIMENT_CONDITIONS:
        raise ValueError("조건은 scratch 또는 coco입니다. 3.3 설정을 확인하세요.")
    results = load_experiment_results()
    results = results.loc[
        results["initialization"].eq(condition) & results["train_seed"].isin(TRAIN_SEEDS)
    ].sort_values("train_seed").copy()
    if require_complete and set(results["train_seed"]) != set(TRAIN_SEEDS):
        section = "3.5" if condition == "scratch" else "3.7"
        raise RuntimeError(f"{condition}의 seed 0·1·2가 모두 필요합니다. {section}의 학습 셀을 완료하세요.")
    for checkpoint in results["best_checkpoint"]:
        if not Path(checkpoint).is_file():
            raise FileNotFoundError("기록된 best.pt가 없습니다. Drive 연결과 해당 run 폴더를 확인하세요.")
    return results


def show_condition_results(condition):
    """조건별 완료 결과를 표시하고 후속 제출에 사용할 표를 반환한다.

    Args:
        condition (str): scratch 또는 coco.

    Returns:
        pandas.DataFrame: 세 seed의 결과와 best.pt 경로.
    """
    results = get_condition_results(condition)
    display(results[[
        "initialization", "train_seed", "best_val_map_75_95", "best_epoch",
        "last_epoch", "total_training_seconds", "best_val_inference_ms_per_image",
        "best_checkpoint",
    ]])
    scores = results["best_val_map_75_95"]
    print(f"{condition} | validation mAP 평균={scores.mean():.4f}, 표본 표준편차={scores.std(ddof=1):.4f}")
    print(f"다음 단계: 4번에서 {condition}의 세 best.pt로 Test 예측·CSV 생성·Kaggle 제출")
    return results


### 3.5 A: 사전학습 없음 seed 0·1·2 학습

아래 셀을 seed 0 → 1 → 2 순서로 한 개씩 실행한다. 각 셀은 최대 100 epoch이다.
완료된 조건·seed의 셀을 다시 실행하면 저장된 결과를 재사용하며 학습을 중복 실행하지 않는다.
**Run All을 사용하지 않는다.** 세 seed가 끝나면 3.6 결과를 확인하고 **4번 제출 단계로 이동**한다.


#### 3.5.1 사전학습 없음 / seed 0


In [ ]:
scratch_seed_0_result = get_or_train_result("scratch", 0)
display(pd.DataFrame([scratch_seed_0_result]))


#### 3.5.2 사전학습 없음 / seed 1


In [ ]:
scratch_seed_1_result = get_or_train_result("scratch", 1)
display(pd.DataFrame([scratch_seed_1_result]))


#### 3.5.3 사전학습 없음 / seed 2


In [ ]:
scratch_seed_2_result = get_or_train_result("scratch", 2)
display(pd.DataFrame([scratch_seed_2_result]))


### 3.6 사전학습 없음 결과 확인 → Kaggle 3건 제출

세 seed의 validation 결과와 best.pt를 확인한다. 제출 조건은 `scratch`로 지정한다.
이후 **4번에서 seed 0·1·2 각각의 CSV를 생성해 Kaggle에 3회 제출**한다.
세 제출을 마친 다음 3.7로 돌아와 COCO 추가 실험을 진행한다.
4.1~4.7을 순서대로 실행해 현재 조건의 제출 파일을 만든다.


In [ ]:
scratch_seed_results = show_condition_results("scratch")
SUBMISSION_CONDITION = "scratch"
submission_runs = scratch_seed_results.copy()
print("여기서 학습을 멈추고 4번으로 이동하세요. scratch CSV 3개를 제출한 뒤 3.7을 실행합니다.")


### 3.7 B: COCO 사전학습 seed 0·1·2 추가 실험

**사전학습 없는 모델의 Kaggle 3건 제출이 끝난 뒤 실행한다.**
앞서 확인한 scratch 성능에 따라 학습 설정을 바꾸지 않고, 초기 가중치만 COCO로 바꿔 같은 seed 세 개를 학습한다.
첫 COCO 셀에서 사전학습 없음의 세 학습 완료 여부를 검사한다. Kaggle 제출 완료 여부는 이 코드가 자동 확인하지 않는다.


#### 3.7.1 COCO 사전학습 / seed 0


In [ ]:
get_condition_results("scratch")  # 사전학습 없는 세 seed 학습 완료 확인
coco_seed_0_result = get_or_train_result("coco", 0)
display(pd.DataFrame([coco_seed_0_result]))


#### 3.7.2 COCO 사전학습 / seed 1


In [ ]:
coco_seed_1_result = get_or_train_result("coco", 1)
display(pd.DataFrame([coco_seed_1_result]))


#### 3.7.3 COCO 사전학습 / seed 2


In [ ]:
coco_seed_2_result = get_or_train_result("coco", 2)
display(pd.DataFrame([coco_seed_2_result]))


### 3.8 COCO 결과 확인 → Kaggle 3건 제출

COCO 세 seed의 validation 결과와 best.pt를 확인하고 제출 조건을 `coco`로 지정한다.
**4번을 다시 실행해 COCO seed 0·1·2의 CSV 3개를 생성하고 Kaggle에 각각 제출**한다.
scratch 제출 CSV를 덮어쓰지 않도록 4번의 파일명에도 초기화 조건과 seed를 포함한다.


In [ ]:
coco_seed_results = show_condition_results("coco")
SUBMISSION_CONDITION = "coco"
submission_runs = coco_seed_results.copy()
print("다음 단계: 4번에서 coco CSV 3개를 생성하고 각각 제출하세요.")


### 3.9 두 조건의 seed별 최종 비교

모든 학습과 제출을 마친 뒤 두 조건을 비교한다. 같은 seed끼리의 validation mAP 차이와
각 조건의 3개 seed 평균·표준편차, 학습·추론 시간을 함께 확인한다.
Kaggle 점수는 4번의 제출 기록에 조건·seed별로 정리한다.


In [ ]:
all_seed_results = pd.concat([
    get_condition_results("scratch"), get_condition_results("coco")
], ignore_index=True)
display(all_seed_results[[
    "initialization", "train_seed", "best_val_map_75_95", "best_epoch",
    "total_training_seconds", "best_val_inference_ms_per_image",
]])
condition_comparison = all_seed_results.groupby("initialization").agg(
    seed_count=("train_seed", "count"),
    val_map_mean=("best_val_map_75_95", "mean"),
    val_map_std=("best_val_map_75_95", "std"),
    training_seconds_mean=("total_training_seconds", "mean"),
    inference_ms_mean=("best_val_inference_ms_per_image", "mean"),
)
display(condition_comparison)
paired_seed_comparison = all_seed_results.pivot(
    index="train_seed", columns="initialization", values="best_val_map_75_95"
)
paired_seed_comparison["coco_minus_scratch"] = (
    paired_seed_comparison["coco"] - paired_seed_comparison["scratch"]
)
display(paired_seed_comparison)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for row in all_seed_results.to_dict("records"):
    history = pd.read_csv(Path(row["run_dir"]) / "metrics.csv")
    label = f"{row['initialization']} / seed {row['train_seed']}"
    axes[0].plot(history["epoch"], history["train_loss"], label=label)
    axes[1].plot(history["epoch"], history["val_map_75_95"], label=label)
for axis, title in zip(axes, ["Train loss", "Validation mAP@[0.75:0.95]"]):
    axis.set(xlabel="Epoch", title=title)
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 4. 조건·시드별 Test 예측 및 Kaggle 제출

**실행 순서:** scratch 세 seed 학습 완료 → **3.6 → 4.1~4.6 → Kaggle 3건 제출 → 4.7 점수 기록** →
3.7의 COCO 세 seed 학습 → **3.8 → 4.1~4.6 재실행 → Kaggle 3건 제출 → 4.7 점수 기록** → 3.9 비교.
한 번에 현재 조건의 CSV 3개를 만든다. 두 조건 합계 6개이며 seed를 섞거나 앙상블하지 않는다.
Test에는 정답 라벨이 없으므로 여기서 mAP를 계산하지 않고 Kaggle에서 나온 점수를 기록한다.
이 절은 새 학습을 시작하지 않고 각 run의 `best.pt`를 사용한다. Kaggle 자동 제출은 실행하지 않는다.


### 4.1 제출 조건·체크포인트·클래스 표 확인

3.6 또는 3.8에서 지정한 `SUBMISSION_CONDITION`을 사용한다. 세 seed의 best.pt·config·클래스 번호표가 필요하다.
제출용 score는 0.001로 낮게 유지한다. 이미지 표시용 0.30과 분리해 낮은 점수 후보도 평가에 포함한다.
모델 내부의 클래스별 NMS를 사용하며 별도 NMS를 중복 적용하지 않는다. 두 조건은 같은 추론 설정을 쓴다.


In [ ]:
import gc
import re
from pill_detection.utils import clip_xywh

if globals().get("SUBMISSION_CONDITION") not in EXPERIMENT_CONDITIONS:
    raise RuntimeError("제출 조건이 없습니다. 먼저 3.6(scratch) 또는 3.8(coco)을 실행하세요.")
submission_runs = get_condition_results(SUBMISSION_CONDITION)
TEST_INFERENCE_BATCH_SIZE = 1
TEST_SCORE_THRESHOLD = 0.001
TEST_NMS_IOU_THRESHOLD = 0.5
TEST_MAX_DETECTIONS = 100
TEST_VIS_SCORE_THRESHOLD = 0.30
TEST_PREVIEW_COUNT = 3
EXPECTED_TEST_IMAGE_COUNT = 842
FORCE_TEST_PREDICTION = False  # True이면 일치하는 캐시가 있어도 다시 추론
SUBMISSIONS_DIR = DRIVE_PROJECT_DIR / "submissions" / EXPERIMENT_ID
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_COLUMNS = [
    "annotation_id", "image_id", "category_id",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score",
]


def mapping_from_config(config):
    """체크포인트 설정의 모델 라벨을 원래 category_id로 되돌린다.

    Args:
        config (dict): 해당 학습 run의 설정.

    Returns:
        dict: 모델 라벨 1..N에서 원래 category_id로 가는 역매핑.
    """
    forward = {int(category): int(label) for category, label in config["category_to_model_label"].items()}
    if len(set(forward.values())) != len(forward) or set(forward.values()) != set(range(1, len(forward) + 1)):
        raise ValueError("클래스 매핑이 1..N의 일대일 대응이 아닙니다. 해당 run의 config.json·class_map.csv를 확인하세요.")
    if int(config["num_classes_including_background"]) != len(forward) + 1:
        raise ValueError("모델 출력 수와 클래스 매핑이 다릅니다. 해당 run의 학습 설정을 확인하세요.")
    return {label: category for category, label in forward.items()}


def verify_submission_runs(runs, condition):
    """현재 조건의 세 seed와 저장된 학습 설정·클래스 표를 확인한다.

    Args:
        runs (pd.DataFrame): 3.6 또는 3.8의 완료 결과.
        condition (str): scratch 또는 coco.

    Returns:
        dict: seed별 학습 설정.
    """
    if (len(runs) != len(TRAIN_SEEDS) or runs["train_seed"].duplicated().any()
            or set(runs["train_seed"]) != set(TRAIN_SEEDS) or not runs["initialization"].eq(condition).all()):
        raise ValueError("현재 조건의 seed 0·1·2가 필요합니다. 3.6 또는 3.8을 다시 확인하세요.")
    configs = {}
    for row in runs.to_dict("records"):
        run_dir, checkpoint_path = Path(row["run_dir"]), Path(row["best_checkpoint"])
        if not re.fullmatch(r"[A-Za-z0-9_-]+", row["run_name"]):
            raise ValueError("run 이름에 경로 문자가 있습니다. 학습 결과표의 run_name을 확인하세요.")
        if checkpoint_path.parent.resolve() != run_dir.resolve() or checkpoint_path.name != "best.pt":
            raise ValueError("best.pt 경로가 run 폴더와 다릅니다. 3번의 학습 결과표를 확인하세요.")
        for path in [checkpoint_path, run_dir / "config.json", run_dir / "class_map.csv"]:
            if not path.is_file() or path.stat().st_size == 0:
                raise FileNotFoundError(f"필수 학습 결과가 없습니다. Drive 연결·run 폴더를 확인하세요: {path}")
        config = json.loads((run_dir / "config.json").read_text())
        if (config["initialization"] != condition or int(config["train_seed"]) != int(row["train_seed"])
                or config["experiment_id"] != EXPERIMENT_ID):
            raise ValueError("조건·seed·실험 ID가 다릅니다. 3번 결과와 4.1의 조건을 확인하세요.")
        reverse = mapping_from_config(config)
        table = pd.read_csv(run_dir / "class_map.csv")
        if table["faster_rcnn_label"].duplicated().any() or table["category_id"].duplicated().any():
            raise ValueError("클래스 번호표가 중복됐습니다. 해당 run의 class_map.csv를 확인하세요.")
        saved_reverse = dict(zip(table["faster_rcnn_label"].astype(int), table["category_id"].astype(int)))
        if saved_reverse != reverse:
            raise ValueError("클래스 번호표와 학습 설정의 매핑이 다릅니다. 해당 run 파일을 확인하세요.")
        configs[int(row["train_seed"])] = config
    return configs


submission_configs = verify_submission_runs(submission_runs, SUBMISSION_CONDITION)
# 조건을 바꿔 4.1을 다시 실행하면 이전 조건의 메모리 결과를 재사용하지 않는다.
for name in ("test_run_results", "submission_exports", "export_manifest"):
    globals().pop(name, None)
print("이번 제출 조건:", SUBMISSION_CONDITION, "| seed:", TRAIN_SEEDS)
print("예측 설정: batch=1, score>=0.001, 모델 내부 NMS IoU=0.5, 최대100개/이미지, AMP=False")
display(submission_runs[["initialization", "train_seed", "best_epoch", "best_val_map_75_95", "best_checkpoint"]])


### 4.2 Test 파일·image_id 확인과 로컬 캐시

현재 제공된 Test 842장의 업로드 완료 여부를 확인한다. 파일명 숫자를 image_id로 사용하며 중복·비숫자 이름은 중단한다.
Drive 원본을 변경하지 않고 1.7의 복사·검증 함수를 재사용해 Colab 디스크에 준비한다.
두 번째 조건에서는 같은 Test 복사본을 재사용한다. 이미지 원본 크기를 좌표 검증용 manifest에 남긴다.


In [ ]:
DRIVE_TEST_IMAGE_DIR = DATA_DIR / "raw" / "test_images"


def collect_test_files(image_dir, expected_count=None):
    """Test 파일명 숫자로 ID를 만들고 누락·중복을 검사한다.

    Args:
        image_dir (Path): Drive Test 이미지 폴더.
        expected_count (int | None): 현재 제공 데이터의 예상 이미지 수.

    Returns:
        pd.DataFrame: file_name·image_id 목록.
    """
    image_dir = Path(image_dir)
    if not image_dir.is_dir():
        raise FileNotFoundError(f"Test 이미지 폴더가 없습니다. Drive 업로드와 0.4 경로를 확인하세요: {image_dir}")
    rows = []
    for path in sorted(image_dir.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in {".png", ".jpg", ".jpeg", ".bmp", ".webp"}:
            continue
        if not re.fullmatch(r"[0-9]+", path.stem):
            raise ValueError(f"파일명에서 숫자 image_id를 만들 수 없습니다. Test 폴더를 확인하세요: {path.name}")
        rows.append({"file_name": path.relative_to(image_dir).as_posix(), "image_id": int(path.stem)})
    if not rows:
        raise ValueError("Test 이미지가 없습니다. 업로드가 끝났는지 확인하세요.")
    frame = pd.DataFrame(rows).sort_values("image_id").reset_index(drop=True)
    if frame["image_id"].duplicated().any():
        raise ValueError("같은 숫자 ID의 Test 파일이 여러 개입니다. 파일명·확장자 중복을 확인하세요.")
    if expected_count is not None and len(frame) != expected_count:
        raise ValueError(f"Test 이미지가 {len(frame)}장입니다. 현재 제공 데이터 {expected_count}장의 업로드 완료 여부를 확인하세요.")
    return frame


test_manifest = collect_test_files(DRIVE_TEST_IMAGE_DIR, EXPECTED_TEST_IMAGE_COUNT)
TEST_IMAGE_DIR, TEST_IMAGE_CONTENT_SHA256 = cache_training_images(
    DRIVE_TEST_IMAGE_DIR, test_manifest["file_name"].tolist(), LOCAL_IMAGE_CACHE_ROOT,
)
sizes = []
for name in test_manifest["file_name"]:
    with Image.open(TEST_IMAGE_DIR / name) as image:
        sizes.append(image.size)
test_manifest[["width", "height"]] = pd.DataFrame(sizes, columns=["width", "height"])
if not (test_manifest[["width", "height"]] > 0).all().all():
    raise ValueError("크기가 0인 Test 이미지가 있습니다. 원본 파일을 확인하세요.")
TEST_PREVIEW_IMAGE_IDS = test_manifest["image_id"].head(TEST_PREVIEW_COUNT).tolist()
print("Test 이미지:", len(test_manifest), "장 | image_id는 파일명 숫자 그대로 사용")
print("Test 이미지 읽기:", TEST_IMAGE_DIR)
display(test_manifest.head())


### 4.3 각 seed의 best.pt로 Test 예측

모델을 한 개씩 복원해 Test 전체를 예측한다. COCO 조건도 이미 학습한 가중치를 읽으므로 사전학습 파일을 다시 내려받지 않는다.
checkpoint는 CPU에 읽고 모델만 GPU로 옮긴다. 추론 batch는 1, FP32·workers 0을 유지한다.
각 run에 `test_predictions.pt`와 `test_inference.json`을 저장한다. 가중치·Test 내용·추론 설정이 같은 캐시만 재사용한다.
순수 모델 추론 시간과 이미지 읽기를 포함한 전체 예측 시간은 구분해서 기록한다.
[torchvision 출력 형식](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn.html)


In [ ]:
class TestImageDataset(Dataset):
    """Test 이미지를 원본 크기 텐서와 파일 ID로 전달한다."""
    def __init__(self, frame, image_dir):
        self.frame = frame.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(self.image_dir / row["file_name"]) as image:
            tensor = to_tensor(image.convert("RGB"))
        return tensor, {"image_id": int(row["image_id"]), "file_name": row["file_name"]}


def atomic_json_save(path, value):
    """JSON을 임시 파일에 기록한 뒤 교체한다.

    Args:
        path (Path): 저장할 파일.
        value (dict): JSON으로 저장할 데이터.
    """
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    try:
        temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def test_prediction_signature(run, config):
    """학습 가중치·Test 내용·추론 설정을 구분할 서명을 만든다.

    Args:
        run (dict): 학습 완료 결과.
        config (dict): 해당 run의 설정.

    Returns:
        dict: 캐시 재사용 여부를 판단할 정보.
    """
    info = Path(run["best_checkpoint"]).stat()
    return {
        "schema_version": 1, "run_name": run["run_name"],
        "experiment_id": EXPERIMENT_ID, "initialization": run["initialization"],
        "train_seed": int(run["train_seed"]),
        "checkpoint_bytes": info.st_size, "checkpoint_mtime_ns": info.st_mtime_ns,
        "test_content_sha256": TEST_IMAGE_CONTENT_SHA256,
        "test_image_ids": test_manifest["image_id"].astype(int).tolist(),
        "test_file_names": test_manifest["file_name"].tolist(),
        "score_threshold": TEST_SCORE_THRESHOLD, "nms_iou": TEST_NMS_IOU_THRESHOLD,
        "max_detections": TEST_MAX_DETECTIONS, "batch_size": TEST_INFERENCE_BATCH_SIZE,
        "precision": "fp32", "amp": False,
        "model_config_sha256": hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest(),
        "inference_environment": RUNTIME_ENVIRONMENT,
    }


def load_model_for_test(run, config):
    """추가 COCO 다운로드 없이 저장된 best.pt로 모델을 복원한다.

    Args:
        run (dict): 학습 완료 결과.
        config (dict): 저장된 학습 설정.

    Returns:
        tuple: 평가 모델과 best checkpoint의 epoch.
    """
    checkpoint = torch.load(run["best_checkpoint"], map_location="cpu", weights_only=True)
    saved = checkpoint.get("config", {})
    for key in ["model", "experiment_id", "initialization", "train_seed", "category_to_model_label",
                "num_classes_including_background", "min_size", "max_size", "normalization"]:
        if saved.get(key) != config.get(key):
            raise ValueError(f"best.pt와 config.json의 {key}가 다릅니다. 해당 run 폴더를 확인하세요.")
    if (saved["environment"]["uv_lock_sha256"] != RUNTIME_ENVIRONMENT["uv_lock_sha256"]
            or saved["normalization"] != "FrozenBatchNorm2d_eps0_both_conditions"):
        raise ValueError("체크포인트의 환경·정규화가 다릅니다. 해당 학습 설정의 환경으로 맞춘 뒤 실행하세요.")
    model = fasterrcnn_resnet50_fpn(
        weights=None, weights_backbone=None,
        num_classes=int(config["num_classes_including_background"]),
        min_size=int(config["min_size"]), max_size=int(config["max_size"]),
    )
    use_common_normalization(model)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    best_epoch = int(checkpoint["epoch"])
    if best_epoch != int(run["best_epoch"]):
        raise ValueError("best.pt의 epoch와 결과표가 다릅니다. 해당 run의 result.json·결과표를 확인하세요.")
    del checkpoint
    model.roi_heads.score_thresh = TEST_SCORE_THRESHOLD
    model.roi_heads.nms_thresh = TEST_NMS_IOU_THRESHOLD
    model.roi_heads.detections_per_img = TEST_MAX_DETECTIONS
    model.requires_grad_(False)
    return model.to(DEVICE).eval(), best_epoch


def validate_prediction_payload(payload, signature):
    """예측 캐시가 현재 run·Test 목록 전체와 대응되는지 확인한다.

    Args:
        payload (dict): CPU 예측 캐시.
        signature (dict): 현재 추론 설정의 서명.
    """
    if payload.get("signature") != signature:
        raise ValueError("예측 캐시 설정이 다릅니다. 4.3을 다시 실행해 현재 설정으로 예측하세요.")
    records = payload.get("predictions", [])
    if [int(item["image_id"]) for item in records] != signature["test_image_ids"]:
        raise ValueError("Test 예측이 누락·중복됐습니다. 4.3을 다시 실행하세요.")
    if [item["file_name"] for item in records] != signature["test_file_names"]:
        raise ValueError("Test 파일명과 예측 순서가 다릅니다. 4.2~4.3을 다시 실행하세요.")
    for record in records:
        if (record["boxes"].ndim != 2 or record["boxes"].shape[1] != 4
                or record["labels"].ndim != 1 or record["scores"].ndim != 1
                or len(record["boxes"]) != len(record["labels"]) or len(record["labels"]) != len(record["scores"])):
            raise ValueError("예측 boxes·labels·scores의 길이 또는 형식이 다릅니다. 해당 run의 예측을 다시 생성하세요.")


def predict_test_run(run, config, force=False):
    """한 seed의 Test 전체를 예측하거나 일치하는 완료 캐시를 재사용한다.

    Args:
        run (dict): 학습 완료 결과.
        config (dict): 해당 run의 설정.
        force (bool): 기존 캐시가 있어도 다시 추론할지 여부.

    Returns:
        dict: 전체 이미지 예측과 시간·설정 정보.
    """
    signature = test_prediction_signature(run, config)
    prediction_path = Path(run["run_dir"]) / "test_predictions.pt"
    if prediction_path.is_file() and not force:
        payload = torch.load(prediction_path, map_location="cpu", weights_only=True)
        if payload.get("signature") == signature:
            validate_prediction_payload(payload, signature)
            print(f"완료 예측 재사용: {run['initialization']} / seed {run['train_seed']}")
            return payload
    print(f"best.pt 불러오기: {run['initialization']} / seed {run['train_seed']}", flush=True)
    load_start = time.perf_counter()
    model, best_epoch = load_model_for_test(run, config)
    model_loading_seconds = time.perf_counter() - load_start
    loader = DataLoader(
        TestImageDataset(test_manifest, TEST_IMAGE_DIR), batch_size=TEST_INFERENCE_BATCH_SIZE,
        shuffle=False, num_workers=0, pin_memory=PIN_MEMORY, collate_fn=detection_collate_fn,
    )
    records, inference_seconds, processed = [], 0.0, 0
    prediction_start = time.perf_counter()
    try:
        with torch.inference_mode():
            for images, metadata in loader:
                device_images = [image.to(DEVICE, non_blocking=PIN_MEMORY) for image in images]
                if DEVICE.type == "cuda":
                    torch.cuda.synchronize()
                start = time.perf_counter()
                outputs = model(device_images)
                if DEVICE.type == "cuda":
                    torch.cuda.synchronize()
                inference_seconds += time.perf_counter() - start
                if len(outputs) != len(metadata):
                    raise RuntimeError("모델 출력 수와 입력 이미지 수가 다릅니다. batch 설정과 모델을 확인하세요.")
                for item, output in zip(metadata, outputs):
                    records.append({**item, "boxes": output["boxes"].detach().cpu(),
                                    "labels": output["labels"].detach().cpu(), "scores": output["scores"].detach().cpu()})
                processed += len(metadata)
                if processed % 100 == 0 or processed == len(test_manifest):
                    print(f"{run['initialization']} seed {run['train_seed']}: {processed}/{len(test_manifest)}장", flush=True)
        elapsed = time.perf_counter() - prediction_start
    finally:
        del model, loader
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    payload = {
        "signature": signature, "best_epoch": best_epoch, "predictions": records,
        "model_loading_seconds": model_loading_seconds,
        "prediction_wall_seconds": elapsed, "inference_seconds": inference_seconds,
        "inference_ms_per_image": inference_seconds / max(processed, 1) * 1000,
        "predicted_images": processed,
    }
    validate_prediction_payload(payload, signature)
    temporary = prediction_path.with_name(prediction_path.name + ".tmp")
    try:
        torch.save(payload, temporary)
        os.replace(temporary, prediction_path)
    finally:
        temporary.unlink(missing_ok=True)
    atomic_json_save(Path(run["run_dir"]) / "test_inference.json", {key: value for key, value in payload.items() if key != "predictions"})
    return payload


RUNTIME_ENVIRONMENT = validate_training_environment()
test_run_results = {}
for run in submission_runs.to_dict("records"):
    seed = int(run["train_seed"])
    test_run_results[seed] = predict_test_run(run, submission_configs[seed], FORCE_TEST_PREDICTION)
display(pd.DataFrame([
    {"initialization": SUBMISSION_CONDITION, "train_seed": seed,
     "test_images": result["predicted_images"], "best_epoch": result["best_epoch"],
     "inference_ms_per_image": result["inference_ms_per_image"],
     "prediction_wall_seconds": result["prediction_wall_seconds"]}
    for seed, result in test_run_results.items()
]))


### 4.4 같은 Test 이미지에서 seed별 예측 확인

같은 이미지 3장을 세 seed에서 비교한다. 추가 추론 없이 저장된 예측을 읽는다.
빨간 bbox는 모델 예측이며 오류 확정 표시가 아니다. Test 정답이 없으므로 초록 정답 박스나 정답률은 표시하지 않는다.
화면에는 score 0.30 이상 상위 20개만 표시하지만 제출 CSV는 4.1의 0.001 기준을 사용한다.


In [ ]:
fig, axes = plt.subplots(len(TRAIN_SEEDS), len(TEST_PREVIEW_IMAGE_IDS),
                         figsize=(6 * len(TEST_PREVIEW_IMAGE_IDS), 5 * len(TRAIN_SEEDS)), squeeze=False)
for row_index, seed in enumerate(TRAIN_SEEDS):
    payload = test_run_results[seed]
    run = submission_runs.loc[submission_runs["train_seed"].eq(seed)].iloc[0].to_dict()
    validate_prediction_payload(payload, test_prediction_signature(run, submission_configs[seed]))
    by_id = {int(record["image_id"]): record for record in payload["predictions"]}
    for column_index, image_id in enumerate(TEST_PREVIEW_IMAGE_IDS):
        record = by_id[int(image_id)]
        reverse = mapping_from_config(submission_configs[seed])
        selected = torch.where(record["scores"] >= TEST_VIS_SCORE_THRESHOLD)[0]
        selected = selected[torch.argsort(record["scores"][selected], descending=True)][:20].tolist()
        labels = [f"{reverse[int(record['labels'][index])]} {float(record['scores'][index]):.2f}" for index in selected]
        canvas = draw_boxes(TEST_IMAGE_DIR / record["file_name"],
                            [xyxy_to_xywh(record["boxes"][index].tolist()) for index in selected],
                            labels, color="red", width=4)
        canvas.thumbnail((1280, 1280))
        axis = axes[row_index, column_index]
        axis.imshow(canvas)
        axis.set_title(f"{SUBMISSION_CONDITION} / seed {seed} / ID {image_id}\nPred {len(selected)} (score>={TEST_VIS_SCORE_THRESHOLD})", fontsize=11)
        axis.axis("off")
fig.suptitle("Test predictions only (red); no ground-truth labels", fontsize=14)
fig.tight_layout()
preview_path = SUBMISSIONS_DIR / f"{SUBMISSION_CONDITION}_test_preview.png"
fig.savefig(preview_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print("예측 확인 이미지:", preview_path)


### 4.5 원래 약 코드·xywh로 제출 표 생성

각 run의 학습 매핑으로 모델 라벨 1..56을 **원래 category_id**로 되돌린다. annotation_id는 CSV마다 1부터 고유하게 부여한다.
모델이 반환한 xyxy는 이미 **원본 이미지 픽셀 좌표**다. 640에서 원본 크기로 다시 확대하지 않고 xywh로만 변환한다.
공용 bbox 함수로 경계를 정리하고, 면적 0·유한하지 않은 예측은 제외하면서 개수를 기록한다.
예측이 없는 이미지는 억지 배경 행을 만들지 않는다. 전체 Test 842장이 추론됐는지는 예측 캐시와 manifest로 확인한다.


In [ ]:
def predictions_to_submission(payload, config, manifest):
    """원본 픽셀 xyxy 예측을 대회 category_id·xywh CSV 행으로 바꾼다.

    Args:
        payload (dict): 전체 Test 예측.
        config (dict): 해당 run의 학습 설정·클래스 역매핑.
        manifest (pd.DataFrame): Test 파일명·ID·원본 크기.

    Returns:
        tuple: 제출 DataFrame과 필터링 통계.
    """
    reverse = mapping_from_config(config)
    images = manifest.set_index("image_id").to_dict("index")
    rows = []
    filtered = {"below_threshold": 0, "nonfinite": 0, "zero_area": 0}
    for record in payload["predictions"]:
        image_id = int(record["image_id"])
        if image_id not in images:
            raise ValueError("Test 목록에 없는 image_id입니다. 4.2~4.3을 다시 확인하세요.")
        info = images[image_id]
        for box, label, score in zip(record["boxes"], record["labels"], record["scores"]):
            values, label, score = box.tolist(), int(label), float(score)
            if label not in reverse:
                raise ValueError(f"알 수 없는 모델 라벨 {label}입니다. 해당 run의 클래스 번호표를 확인하세요.")
            if not np.isfinite(values).all() or not np.isfinite(score):
                filtered["nonfinite"] += 1
                continue
            if not 0 <= score <= 1:
                raise ValueError("예측 score가 0~1 범위 밖입니다. 모델 출력과 체크포인트를 확인하세요.")
            if score < TEST_SCORE_THRESHOLD:
                filtered["below_threshold"] += 1
                continue
            bbox = clip_xywh(xyxy_to_xywh(values), int(info["width"]), int(info["height"]))
            if bbox[2] <= 0 or bbox[3] <= 0:
                filtered["zero_area"] += 1
                continue
            rows.append({"image_id": image_id, "category_id": reverse[label],
                         "bbox_x": bbox[0], "bbox_y": bbox[1], "bbox_w": bbox[2], "bbox_h": bbox[3], "score": score})
    frame = pd.DataFrame(rows, columns=SUBMISSION_COLUMNS[1:])
    if frame.empty:
        raise ValueError("제출할 예측이 한 개도 없습니다. 4.4의 이미지·모델·score 설정을 확인하세요.")
    frame = frame.sort_values(["image_id", "score"], ascending=[True, False], kind="stable").reset_index(drop=True)
    frame.insert(0, "annotation_id", np.arange(1, len(frame) + 1, dtype=np.int64))
    frame[["annotation_id", "image_id", "category_id"]] = frame[["annotation_id", "image_id", "category_id"]].astype("int64")
    return frame[SUBMISSION_COLUMNS], filtered


submission_exports = {}
for run in submission_runs.to_dict("records"):
    seed = int(run["train_seed"])
    payload = test_run_results[seed]
    signature = test_prediction_signature(run, submission_configs[seed])
    validate_prediction_payload(payload, signature)
    frame, filtered = predictions_to_submission(payload, submission_configs[seed], test_manifest)
    submission_exports[seed] = {"run": run, "signature": signature, "frame": frame, "filtered": filtered}
    print(f"{SUBMISSION_CONDITION} / seed {seed}: {len(frame)}개 예측 | 필터링 {filtered}")
    display(frame.head(3))


### 4.6 제출 파일 검증·Drive 저장

컬럼 순서, 숫자·결측, annotation_id 고유성, image_id·원래 약 코드, score 0..1, bbox 경계를 검사한다.
저장한 CSV도 다시 읽어 검사한다. Drive의 `submissions/<실험ID>/`에 `scratch_s0_월일_시간.csv`처럼 저장한다.
seed별 제출 CSV 3개와 설정·검증 JSON을 보관한다. 조건별 파일 목록과 전체 Test manifest는 관리용이며 제출하지 않는다.
대회 컬럼: `annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score`.
[대회 제출 규칙](../../RULE.md)


In [ ]:
def validate_submission(frame, config, manifest):
    """대회 컬럼·ID·좌표·점수와 원본 이미지 경계를 검사한다.

    Args:
        frame (pd.DataFrame): 제출 표.
        config (dict): 해당 run의 설정.
        manifest (pd.DataFrame): 전체 Test 이미지와 원본 크기.

    Returns:
        dict: 예측 행 수·예측 이미지 수·예측 없는 이미지 수.
    """
    if list(frame.columns) != SUBMISSION_COLUMNS or frame.empty:
        raise ValueError("제출 컬럼 또는 행이 잘못됐습니다. 4.5를 다시 실행하세요.")
    if frame.isna().any().any() or not np.isfinite(frame.to_numpy(dtype=float)).all():
        raise ValueError("결측 또는 무한대가 있습니다. 예측 출력과 4.5 변환을 확인하세요.")
    for name in ["annotation_id", "image_id", "category_id"]:
        if not np.equal(frame[name].to_numpy(), np.floor(frame[name].to_numpy())).all():
            raise ValueError(f"{name}이 정수가 아닙니다. 4.5의 ID 변환을 확인하세요.")
    if frame["annotation_id"].duplicated().any() or not frame["annotation_id"].gt(0).all():
        raise ValueError("annotation_id가 중복되거나 0 이하입니다. 4.5를 다시 실행하세요.")
    if not frame["image_id"].isin(manifest["image_id"]).all():
        raise ValueError("제출 image_id가 Test 파일명 목록과 다릅니다. 4.2와 변환 코드를 확인하세요.")
    allowed_categories = set(mapping_from_config(config).values())
    if not frame["category_id"].isin(allowed_categories).all():
        raise ValueError("원래 category_id로 복원되지 않았습니다. 해당 run의 역매핑을 확인하세요.")
    if not frame["score"].between(0, 1).all():
        raise ValueError("score가 0~1 범위 밖입니다. 예측 출력을 확인하세요.")
    joined = frame.merge(manifest[["image_id", "width", "height"]], on="image_id", how="left", validate="many_to_one")
    valid_boxes = (joined["bbox_x"].ge(0) & joined["bbox_y"].ge(0)
                   & joined["bbox_w"].gt(0) & joined["bbox_h"].gt(0)
                   & (joined["bbox_x"] + joined["bbox_w"]).le(joined["width"] + 1e-4)
                   & (joined["bbox_y"] + joined["bbox_h"]).le(joined["height"] + 1e-4))
    if not valid_boxes.all():
        raise ValueError("이미지 경계 밖이거나 면적이 0인 bbox가 있습니다. 4.5 좌표 변환을 확인하세요.")
    image_count = frame["image_id"].nunique()
    return {"prediction_rows": len(frame), "images_with_predictions": image_count,
            "images_without_predictions": len(manifest) - image_count}


def save_validated_submission(item, config):
    """검증한 CSV와 추론·파일 정보를 저장하고 다시 읽어 확인한다.

    Args:
        item (dict): 4.5에서 생성한 run·서명·제출 표.
        config (dict): 해당 학습 run 설정.

    Returns:
        dict: 제출 파일 목록에 넣을 한 행.
    """
    run, frame = item["run"], item["frame"]
    if item["signature"] != test_prediction_signature(run, config):
        raise ValueError("CSV 생성 후 예측 조건이 바뀌었습니다. 4.3~4.5를 다시 실행하세요.")
    stats = validate_submission(frame, config, test_manifest)
    destination = SUBMISSIONS_DIR / f"{run['run_name']}.csv"
    temporary = destination.with_name(destination.name + ".tmp")
    try:
        frame.to_csv(temporary, index=False)
        reloaded = pd.read_csv(temporary)
        validate_submission(reloaded, config, test_manifest)
        if len(reloaded) != len(frame):
            raise ValueError("CSV 재읽기 행 수가 다릅니다. 저장 공간과 파일 상태를 확인하세요.")
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)
    digest = file_sha256(destination)
    inference = test_run_results[int(run["train_seed"])]
    export_row = {
        "experiment_id": EXPERIMENT_ID, "initialization": run["initialization"],
        "train_seed": int(run["train_seed"]), "run_name": run["run_name"],
        "best_epoch": int(run["best_epoch"]), "val_map_75_95": float(run["best_val_map_75_95"]),
        "test_images": len(test_manifest), **stats,
        "inference_ms_per_image": inference["inference_ms_per_image"],
        "csv_path": str(destination), "csv_sha256": digest,
    }
    atomic_json_save(destination.with_suffix(".json"), {
        **export_row, "inference_signature": item["signature"], "filtered": item["filtered"],
        "test_predictions_path": str(Path(run["run_dir"]) / "test_predictions.pt"),
    })
    return export_row


export_manifest = pd.DataFrame([
    save_validated_submission(submission_exports[seed], submission_configs[seed]) for seed in TRAIN_SEEDS
])
export_manifest.to_csv(SUBMISSIONS_DIR / f"{SUBMISSION_CONDITION}_files.csv", index=False)
test_manifest.to_csv(SUBMISSIONS_DIR / "test_manifest.csv", index=False)
display(export_manifest[["initialization", "train_seed", "prediction_rows", "images_without_predictions", "csv_path"]])
print("검증 완료. Kaggle에 올릴 파일은 위 목록의 seed별 CSV 3개입니다.")
print("파일 목록·test_manifest·점수 기록용 CSV는 Kaggle 제출 파일이 아닙니다.")


### 4.7 Kaggle 웹 제출과 점수 기록

1. 4.6에 표시된 seed별 CSV 3개를 Drive에서 다운로드한다.
2. Kaggle 대회의 **Submit Predictions**에서 파일을 한 개씩 올리고 제출한다.
3. 점수가 나오면 아래 `KAGGLE_PUBLIC_SCORES`의 seed별 None을 실제 점수로 바꿔 이 셀을 실행한다.

이 셀은 Kaggle에 파일을 업로드하거나 제출하지 않는다. 점수를 입력하지 않으면 csv_ready 상태로 파일 목록만 기록한다.
점수 기록 시각을 남기고, 두 조건·여섯 seed 결과를 같은 `kaggle_results.csv`에 누적한다. None은 기존 점수를 지우지 않는다.
scratch 세 제출 후 COCO 추가 실험을 진행하고, COCO 세 제출 후 3.9에서 validation 변동과 함께 비교한다.


In [ ]:
# Kaggle 웹에서 현재 조건의 CSV 3개를 각각 제출한 뒤, 표시된 점수를 적는다.
# None은 아직 점수를 입력하지 않았다는 뜻이며 이전에 기록한 점수를 지우지 않는다.
KAGGLE_PUBLIC_SCORES = {0: None, 1: None, 2: None}


def record_kaggle_scores(exports, scores):
    """수동 제출 후 확인한 Kaggle 점수를 조건·seed·run별로 보관한다.

    Args:
        exports (pd.DataFrame): 4.6에서 검증·저장한 파일 목록.
        scores (dict): seed별 Kaggle 공개 점수. 미입력은 None.

    Returns:
        pd.DataFrame: 두 조건의 누적 제출 파일·점수 기록.
    """
    log_path = SUBMISSIONS_DIR / "kaggle_results.csv"
    rows = {}
    if log_path.is_file():
        for row in pd.read_csv(log_path, dtype={"experiment_id": str}).to_dict("records"):
            rows[row["run_name"]] = row
    for entry in exports.to_dict("records"):
        path = Path(entry["csv_path"])
        if entry["experiment_id"] != EXPERIMENT_ID or not path.is_file() or file_sha256(path) != entry["csv_sha256"]:
            raise ValueError("제출 파일이 없거나 저장 후 변경됐습니다. 4.6 검증을 다시 실행하세요.")
        previous = rows.get(entry["run_name"], {})
        value = scores.get(int(entry["train_seed"]))
        score = previous.get("kaggle_public_score", None)
        recorded_at = previous.get("score_recorded_at", None)
        if value is not None:
            score = float(value)
            if not np.isfinite(score) or not 0 <= score <= 1:
                raise ValueError("Kaggle 점수는 0~1의 숫자로 입력하세요. 웹에 표시된 원래 점수를 확인하세요.")
            recorded_at = datetime.now().isoformat()
        rows[entry["run_name"]] = {**entry, "kaggle_public_score": score,
                                  "score_recorded_at": recorded_at,
                                  "status": "score_recorded" if pd.notna(score) else "csv_ready"}
    output = pd.DataFrame(rows.values()).sort_values(["initialization", "train_seed", "run_name"])
    temporary = log_path.with_name(log_path.name + ".tmp")
    try:
        output.to_csv(temporary, index=False)
        os.replace(temporary, log_path)
    finally:
        temporary.unlink(missing_ok=True)
    print("Kaggle 기록:", log_path)
    return output


kaggle_results = record_kaggle_scores(export_manifest, KAGGLE_PUBLIC_SCORES)
display(kaggle_results[["initialization", "train_seed", "val_map_75_95", "kaggle_public_score", "status", "csv_path"]])
if SUBMISSION_CONDITION == "scratch":
    print("scratch CSV 3개를 웹에서 제출한 뒤 3.7로 돌아가 COCO seed 0·1·2를 학습하세요.")
else:
    print("coco CSV 3개를 웹에서 제출한 뒤 3.9의 조건별·seed별 비교를 확인하세요.")
